# MemoRizz memory types in action

## One use case, thirteen memory types, observable before/after effects

In this notebook we build **Nova**, an on-call support agent for a fictional
payments team. Maya owns `checkout-api`; the service has just become slow after
a deployment. Maya wants incident updates kept under five bullets and wants to
be paged before executives are emailed.

A useful support agent must do more than retain chat. Across one continuous
incident it needs to remember Maya's preferences, preserve a compact history,
maintain a stable operating persona, know who owns the service, retrieve the
approved runbook, check live health, audit the tool result, record what procedure
actually ran, apply a reviewed reusable procedure, use per-turn page context,
avoid repeated model calls, hand work to another agent, and survive a restart.

Every section follows the same experiment:

1. ask Nova a question **before** the relevant memory exists;
2. add the memory through MemoRizz's normal API;
3. ask again and inspect both the answer and the stored/runtime evidence;
4. state what the memory does *not* guarantee.

The notebook installs MemoRizz as a normal package. GPT-5.5 supplies the
reasoning and tool use, while OpenAI `text-embedding-3-small` supplies every
semantic vector. A small tracking wrapper records only assembled prompts and
call counts. Assertions therefore prove which memory evidence reached the
model while allowing natural-language answers to vary from run to run.


## The continuity we are building

Memory types are complementary representations, not interchangeable buckets.
The same incident moves through them in a deliberate order:

```mermaid
flowchart LR
  A[No memory\nunknown preference] --> B[Conversation\nwhat Maya said]
  B --> C[Summary\ncompressed history]
  C --> D[Persona\nhow Nova behaves]
  D --> E[Entity + knowledge\nwho/what is true]
  E --> F[Toolbox + tool log\nwhat Nova can do / did]
  F --> G[Workflow + Skillbox\nrepeatable procedure]
  G --> H[Short-term + cache\ncurrent context and reuse]
  H --> I[Shared memory\ncoordinated hand-off]
  I --> J[MemAgent\nrestartable agent]
```

Two rules prevent most memory-system mistakes:

- **Storage is not context.** A record can be durable without being sent to the
  model on every turn. MemoRizz retrieves, scopes, deduplicates, budgets, and
  renders selected evidence.
- **Memory type is a semantic contract.** Conversation answers “what happened,”
  entity memory answers “what is currently known about this object,” a tool log
  answers “what action ran,” and a workflow answers “what sequence occurred.”
  Putting all four into a single transcript loses update and governance rules.

## Complete taxonomy used in the incident

MemoRizz 0.6 exposes thirteen `MemoryType` values. This notebook exercises every
one, including the types that are runtime-owned or operational rather than
ordinary retrievable knowledge.

| Family | `MemoryType` | What its unit means in this use case | Normal owner |
|---|---|---|---|
| Episodic | `CONVERSATION_MEMORY` | Maya and Nova's scoped turns | `MemAgent.run()` |
| Episodic | `SUMMARIES` | A compact, source-linked incident history | summarization lifecycle |
| Semantic | `PERSONAS` | Nova's stable, versioned operating identity | application/reviewed update |
| Semantic | `ENTITY_MEMORY` | Structured facts about `checkout-api` | application or governed entity tools |
| Semantic | `KNOWLEDGE_BASE` | Chunked incident policy and SLO passages | ingestion pipeline |
| Procedural | `TOOLBOX` | Trusted callable metadata and strict schema | host application |
| Procedural | `WORKFLOW_MEMORY` | The actual tool trajectory for each triage run | tool loop |
| Procedural | `SKILLBOX` | A reviewed reusable procedure for live-health triage | host authoring and review |
| Working | `SHORT_TERM_MEMORY` | Runtime-only current-page/task context | context assembler |
| Working | `SEMANTIC_CACHE` | A fresh, fingerprinted reusable answer | cache policy |
| Social | `SHARED_MEMORY` | Typed command/status/report blackboard | orchestrator and participants |
| Operational | `TOOL_LOG` | Full result and audit metadata for a tool call | tool loop |
| Operational | `MEMAGENT` | Nova's durable configuration and memory attachments | save/load lifecycle |

`SHORT_TERM_MEMORY` is intentionally special: the public lesson is to pass
per-turn `context=...` and inspect usage, not to append arbitrary records to a
mutable scratchpad. A persistent row count of zero can therefore be the correct
result.

# Part 0 · Installed package, models, Oracle, and policies

## 0.1 · Install and verify the runtime

The original notebook is an Oracle AI Database field guide, so this revision
keeps the real `OracleProvider`. It uses the course's dedicated `MEMORIZZ`
schema, a unique scope for every run, and verified cleanup at the end.

Reasoning uses the real OpenAI [`gpt-5.5`](https://developers.openai.com/api/docs/models/gpt-5.5)
model through the Responses API with low reasoning effort. We cap output length
because the lesson needs concise support answers rather than long essays.

Retrieval uses the real OpenAI
[`text-embedding-3-small`](https://developers.openai.com/api/docs/models/text-embedding-3-small)
model. We request 256-dimensional vectors so the hosted representation matches
the course's Oracle `VECTOR(256, FLOAT32)` columns. The same embedding manager
is shared by knowledge, entities, workflows, skills, queries, and cache entries;
mixing vector spaces would make similarity scores meaningless.

Requirements are a Python notebook kernel, the course Oracle container, and
`OPENAI_API_KEY` in the environment or the local MemoRizz `.env`. The key is
loaded at runtime and is never printed or stored in notebook metadata.

The install is deliberately ordinary: `%pip install --upgrade memorizz` installs
the package when absent and upgrades it when an older release is present. The
notebook never adds the checkout's `src` directory to `sys.path` and never uses
an editable or source-tree import.


### Install or upgrade MemoRizz

This is the only package-install command in the lesson. `--upgrade` asks `pip`
for the newest compatible MemoRizz release while leaving already-satisfied
dependencies alone; there is no forced reinstall of the environment.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `%pip install` | package | `memorizz` | Installs from the package index, not from the local `src` tree. |
| `%pip install` | `--upgrade` | enabled | Installs when missing and upgrades an existing older release. |


In [9]:
%pip install --upgrade memorizz



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Import the installed public API

Only public MemoRizz imports and the few documented procedural classes used for audit are loaded. There is no `sys.path.insert`, editable install, or import from `src`.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `memorizz` | `import path` | installed package | The next block verifies its physical location. |
| `EmbeddingManager` | `provider` | configured later | Creates real hosted vectors through MemoRizz's embedding abstraction. |
| `OpenAI` | `api_mode` | configured later | Supplies real GPT reasoning and tool calls. |
| `OracleProvider` | `environment` | configured later | Persists every durable memory representation. |

In [10]:
import json
import os
import re
import uuid
from collections import Counter
from importlib.metadata import version
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display

import memorizz
from memorizz import (
    ContextPolicy, EntityMemory, KnowledgeBase, LocalOracleRuntime,
    MemAgent, MemoryType, OracleProvider, Persona, RetrievalPolicy,
    RoleType, SharedMemory, Toolbox, ToolResultPolicy, governed_tool,
)
from memorizz.embeddings import EmbeddingManager, set_global_embedding_manager
from memorizz.llms.openai import OpenAI
from memorizz.long_term.procedural.skillbox import Skill, SkillStatus
from memorizz.long_term.procedural.workflow.canonicalization import (
    aggregate_trajectory_stats,
)


### Load configuration and prove package provenance

Secrets remain in environment variables. The printed report contains booleans and paths, never credential values.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `load_dotenv` | `override` | `False` | A caller-provided environment value wins over the course `.env`. |
| `Path.resolve` | `package path` | `memorizz.__file__` | Proves which physical package Python imported. |
| Oracle environment | `ORACLE_USER / PASSWORD / DSN` | course-scoped defaults | Keeps the run isolated from other development schemas. |
| `env_flag` | `MEMORIZZ_TUTORIAL_KEEP_DATA` | default `False` | Cleanup is the safe default. |

In [11]:
MEMORIZZ_PROJECT = Path(
    os.getenv("MEMORIZZ_REPO", "/Users/richmondalake/Desktop/memorizz")
).resolve()
env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file, override=False)
if (MEMORIZZ_PROJECT / ".env").exists():
    load_dotenv(MEMORIZZ_PROJECT / ".env", override=False)

package_path = Path(memorizz.__file__).resolve()
source_root = (MEMORIZZ_PROJECT / "src").resolve()
assert "site-packages" in package_path.parts
assert source_root not in package_path.parents

os.environ["ORACLE_USER"] = os.getenv("MEMORIZZ_TUTORIAL_ORACLE_USER", "MEMORIZZ")
os.environ["ORACLE_PASSWORD"] = os.getenv(
    "MEMORIZZ_TUTORIAL_ORACLE_PASSWORD", "MemorizzPwd_2026"
)
os.environ["ORACLE_DSN"] = os.getenv(
    "MEMORIZZ_TUTORIAL_ORACLE_DSN", "localhost:1521/FREEPDB1"
)
print({"memorizz_version": version("memorizz"), "imported_from": str(package_path)})


{'memorizz_version': '0.6.3', 'imported_from': '/Users/richmondalake/opt/anaconda3/envs/oracle_demos/lib/python3.11/site-packages/memorizz/__init__.py'}


### Create isolated identifiers for this run

Every provider read and write is scoped by identifiers owned by the host application. A random run suffix prevents one execution from contaminating another.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `os.getenv` | `MEMORIZZ_TUTORIAL_RUN_ID` | random fallback | Allows an explicit repeatable ID while defaulting to isolation. |
| `memory_id` | `MEMORY_ID` | `nova-support-<run>` | Primary content boundary. |
| `user_id` | `USER_ID` | `maya-<run>` | Tenant/user boundary. |
| `thread_id` | `THREAD_ID` | `checkout-incident` | Conversation boundary within the user scope. |

In [12]:
def env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().lower() in {
        "1", "true", "yes", "on"
    }


RUN_ID = os.getenv("MEMORIZZ_TUTORIAL_RUN_ID", uuid.uuid4().hex[:10])
KEEP_DATA = env_flag("MEMORIZZ_TUTORIAL_KEEP_DATA")
EMBEDDING_DIMENSIONS = int(
    os.getenv("MEMORIZZ_TUTORIAL_EMBEDDING_DIMENSIONS", "256")
)
OPENAI_EMBEDDING_MODEL = os.getenv(
    "MEMORIZZ_TUTORIAL_EMBEDDING_MODEL", "text-embedding-3-small"
)
OPENAI_MODEL, OPENAI_REASONING_EFFORT = "gpt-5.5", "low"
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required in the environment or local .env")

MEMORY_ID, WORKING_MEMORY_ID = f"nova-support-{RUN_ID}", f"nova-working-{RUN_ID}"
USER_ID, OTHER_USER_ID = f"maya-{RUN_ID}", f"other-user-{RUN_ID}"
THREAD_ID, CACHE_THREAD_ID = "checkout-incident", "sev1-policy-cache"
KB_NAMESPACE, ENTITY_ID = f"checkout-runbook-{RUN_ID}", f"checkout-api-{RUN_ID}"
SHARED_WORKFLOW_ID = f"checkout-escalation-{RUN_ID}"
print({"run_id": RUN_ID, "keep_data": KEEP_DATA, "openai_key_available": True})


{'run_id': '0b6953f061', 'keep_data': False, 'openai_key_available': True}


## 0.2 · Real OpenAI embeddings

`EmbeddingManager` constructs MemoRizz's native OpenAI embedding provider. The
probe below makes a genuine API request, checks the returned dimensionality,
and shows only safe metadata plus the vector norm—not credentials or raw text.
Later batch and single-item calls are memoized by MemoRizz within this process,
but every previously unseen text is embedded by the hosted model.

Reduced dimensions are not a local projection: the `dimensions` argument is
sent to OpenAI's embeddings endpoint. Pinning model and dimensions is a storage
contract. Changing either after vectors have been persisted requires a planned
re-embedding migration.

### Embedding model parameters

The same manager is passed to Oracle and registered globally so every semantic memory type occupies one vector space.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `EmbeddingManager` | `provider` | `openai` | Selects MemoRizz's hosted OpenAI embedding provider. |
| `EmbeddingManager` | `model` | `text-embedding-3-small` | Pins the semantic model used for storage and queries. |
| `EmbeddingManager` | `dimensions` | `256` | Matches Oracle's `VECTOR(256, FLOAT32)` schema. |
| `get_embedding` | `text` | incident probe | Makes a real API call and validates the vector contract. |

In [13]:
embeddings = EmbeddingManager(
    provider="openai",
    config={
        "model": OPENAI_EMBEDDING_MODEL,
        "dimensions": EMBEDDING_DIMENSIONS,
    },
)
set_global_embedding_manager(embeddings)
embedding_probe = embeddings.get_embedding(
    "checkout-api latency incident escalation procedure"
)
embedding_norm = sum(value * value for value in embedding_probe) ** 0.5
embedding_effect = {
    "provider": embeddings.get_provider_info()["provider"],
    "model": embeddings.get_default_model(),
    "dimensions_requested": embeddings.get_dimensions(),
    "dimensions_returned": len(embedding_probe),
    "unit_norm": round(embedding_norm, 6),
    "hosted_api_vector": True,
}
print(embedding_effect)
assert embedding_effect["provider"] == "openai"
assert len(embedding_probe) == EMBEDDING_DIMENSIONS
assert 0.99 <= embedding_norm <= 1.01


{'provider': 'openai', 'model': 'text-embedding-3-small', 'dimensions_requested': 256, 'dimensions_returned': 256, 'unit_norm': 0.999999, 'hosted_api_vector': True}


## 0.3 · A tracked GPT-5.5 model that exposes context causality

`TrackedOpenAI` is a thin, non-mocking wrapper around MemoRizz's OpenAI
provider. Every answer, summary, and tool decision still comes from GPT-5.5.
The wrapper records the messages passed to `generate()` and counts calls; it
never records credentials.

This matters for a before/after lesson. Natural language is non-deterministic,
so asserting an exact sentence would be fragile. Instead, each section asserts
that relevant evidence was absent from the *before* prompt, present in the
*after* prompt, correctly scoped, and backed by the expected provider record.
We display GPT's answers as the user-visible consequence of that causal change.


### Tracked GPT provider

Instrumentation records prompts and call counts only. `super().generate(...)` still performs every real GPT request.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `OpenAI.__init__` | `model` | `gpt-5.5` | Uses the requested reasoning model. |
| `OpenAI.__init__` | `reasoning_effort` | `low` | Keeps the teaching run responsive while retaining reasoning/tool support. |
| `OpenAI.__init__` | `api_mode` | `responses` | Uses the Responses API execution path. |
| `OpenAI.__init__` | `max_completion_tokens` | `1200` | Bounds each teaching answer. |

In [14]:
class TrackedOpenAI(OpenAI):
    """Real GPT provider with prompt/call instrumentation for this lesson."""

    def __init__(self):
        super().__init__(
            model=OPENAI_MODEL,
            reasoning_effort=OPENAI_REASONING_EFFORT,
            api_mode="responses",
            max_completion_tokens=1200,
        )
        self.calls = self.text_calls = 0
        self.prompts, self.text_prompts = [], []

    @staticmethod
    def _message_text(messages):
        values = []
        for message in messages or []:
            if not isinstance(message, dict):
                continue
            content = message.get("content")
            if isinstance(content, str):
                values.append(content)
            else:
                values.append(json.dumps(content, ensure_ascii=False, default=str))
        return "\n".join(values)

    def generate(self, messages, tools=None, tool_choice="auto"):
        self.calls += 1
        user_values = [
            str(item.get("content") or "")
            for item in messages or []
            if isinstance(item, dict) and item.get("role") == "user"
        ]
        tool_names = sorted(
            item.get("function", {}).get("name")
            for item in tools or []
            if isinstance(item, dict) and item.get("function", {}).get("name")
        )
        self.prompts.append(
            {
                "kind": "agent_turn",
                "text": self._message_text(messages),
                "latest_user": user_values[-1] if user_values else "",
                "tool_names": tool_names,
            }
        )
        return super().generate(messages, tools=tools, tool_choice=tool_choice)

    def generate_text(self, prompt, instructions=None):
        self.text_calls += 1
        self.text_prompts.append(
            {"prompt": str(prompt), "instructions": str(instructions or "")}
        )
        return super().generate_text(prompt, instructions=instructions)


### Model factory and safe configuration probe

A factory gives each demonstration its own call counters while sharing the same real provider configuration.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `make_model` | `—` | new `TrackedOpenAI` | Prevents cache and short-term experiments from sharing counters. |
| `get_config` | `—` | safe provider metadata | Displays model/API mode without credentials. |
| `get_context_window_tokens` | `—` | provider capability | Lets later context statistics be interpreted. |

In [15]:
def make_model():
    return TrackedOpenAI()


probe_model = make_model()
print({
    **probe_model.get_config(),
    "context_window_tokens": probe_model.get_context_window_tokens(),
    "credentials_recorded": False,
})


{'provider': 'openai', 'model': 'gpt-5.5', 'api_mode': 'responses', 'reasoning_effort': 'low', 'context_window_tokens': 1050000, 'credentials_recorded': False}


## 0.4 · Start Oracle and verify the memory substrate

Provider preflight is part of correctness. A valid Python object is not enough:
the PDB must be open, vector columns must match the embedding dimension, and the
provider must advertise scoped search. `index_policy="lazy"` creates optional
accelerators when each store is first used; exact vector distance remains the
correctness path.

### Oracle runtime and provider parameters

The runtime check proves the database is ready; provider preflight proves the schema and vector configuration are compatible.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `LocalOracleRuntime.from_env` | `provision_if_missing` | `False` | Never creates infrastructure implicitly during a lesson. |
| `LocalOracleRuntime.from_env` | `container_name` | course container | Targets the intended local Oracle instance. |
| `OracleProvider.from_env` | `index_policy` | `lazy` | Creates optional vector indexes only when a store is first used. |
| `OracleProvider.from_env` | `in_database_embedding` | `False` | Uses the real OpenAI manager configured above. |
| `OracleProvider.from_env` | `embedding_provider` | `embeddings` | Keeps stored and query vectors in the same space. |

In [16]:
ORACLE_CONTAINER = os.getenv(
    "MEMORIZZ_TUTORIAL_ORACLE_CONTAINER", "acme-oracle-free"
)
runtime = LocalOracleRuntime.from_env(
    provision_if_missing=False,
    container_name=ORACLE_CONTAINER,
)
runtime_report = runtime.ensure_ready()

provider = OracleProvider.from_env(
    index_policy="lazy",
    in_database_embedding=False,
    embedding_provider=embeddings,
)
preflight = provider.preflight()
embedding_report = preflight.get("embedding") or {}
safe_preflight = {
    "ok": preflight.get("ok"),
    "database_product": preflight.get("database_product"),
    "version_full": preflight.get("version_full"),
    "pdb": preflight.get("pdb"),
    "pdb_open_state": preflight.get("pdb_open_state"),
    "embedding_model": embedding_report.get("model"),
    "embedding_dimensions": embedding_report.get("dimensions"),
    "index_policy": preflight.get("index_policy"),
    "diagnostics": preflight.get("diagnostics"),
    "runtime": {
        key: runtime_report.get(key)
        for key in ("ok", "container", "state", "action")
    },
}
print(safe_preflight)
print("Capabilities:", provider.memory_capabilities().to_dict())

if not preflight.get("ok"):
    raise RuntimeError("Oracle preflight failed; inspect diagnostics above.")
provider.validate_vector_schema_dimensions(EMBEDDING_DIMENSIONS)
assert len(MemoryType) == 13


{'ok': True, 'database_product': 'Oracle AI Database 26ai Free', 'version_full': '23.26.2.0.0', 'pdb': 'FREEPDB1', 'pdb_open_state': 'READ WRITE', 'embedding_model': 'text-embedding-3-small', 'embedding_dimensions': 256, 'index_policy': 'lazy', 'diagnostics': ['Could not inspect Oracle VECTOR dimensions for schema MEMORIZZ: DPY-4011: the database or network closed the connection\nHelp: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpy-4011'], 'runtime': {'ok': True, 'container': 'acme-oracle-free', 'state': 'running', 'action': 'none'}}
Capabilities: {'provider': 'OracleProvider', 'batch_store': False, 'transactional_batch': False, 'scoped_search': True, 'result_scores': True, 'provenance': True, 'native_vector_search': True, 'native_hybrid_search': False}


## 0.5 · Configure the same agent before adding useful memory

Nova is configured with every context-relevant memory partition so we can
observe them in one lifecycle, but the stores are initially empty. `MEMAGENT`
is deliberately excluded from prompt-context types: it is the operational
save/load record demonstrated near the end. Explicit `memory_types` overrides
application-mode defaults; a production application should enable only the
types its workflow needs.

We also make retrieval scope explicit:

- conversation recall is restricted to the current thread;
- knowledge recall is restricted to this lesson's runbook namespace;
- every call carries a host-owned `memory_id`, `user_id`, and `thread_id`.

Skill retrieval is enabled, but the Skillbox starts empty. Section 10 will add
one host-reviewed procedure directly; automated continual learning is outside
the scope of this memory-types notebook. Progressive tool disclosure remains
off until the Toolbox section so early comparisons vary one layer at a time.

### Why these policies are separate

| Policy | Controls | Does not control | Why it exists here |
|---|---|---|---|
| `RetrievalPolicy` | eligible scopes, candidates, result budget, parent dedupe | whether a tool may execute | prevents cross-thread/namespace leakage and retains both needed runbook chunks |
| `ContextPolicy` | progressive tool disclosure and per-turn tool budget | retention of memory rows | keeps irrelevant tools out of early baseline prompts |
| `ToolResultPolicy` | inline versus durable-pointer tool results | permission to execute the tool | preserves full audit data without repeatedly sending 24 diagnostics to GPT |
| Skill retrieval config | active-skill top-k and similarity threshold | authoring or approval of skills | prevents every stored procedure from becoming prompt instructions |

Security scope, prompt size, capability exposure, and instruction authority are
different concerns; keeping their policies separate makes each effect testable.


### Retrieval, skill-retrieval, and tool-result policies

Retrieval stays tenant/thread/namespace scoped. Long live telemetry is
offloaded, while knowledge excerpts and expansion-tool outputs remain inline.
Skill retrieval considers at most two active procedures and requires a semantic
match; it never changes a skill's review status.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `RetrievalPolicy` | `conversation_scope` | `thread` | Only the current incident thread is eligible. |
| `RetrievalPolicy` | `knowledge_base_scope` | `namespace` | Restricts retrieval to this runbook. |
| `RetrievalPolicy` | `candidate_limit / max_items` | `6 / 5` | Bounds vector candidates and prompt items separately. |
| `RetrievalPolicy` | `dedupe_parent_sources` | `False` | Keeps both sibling chunks containing the SLO and SEV threshold. |
| skill retrieval | `top_k / min_similarity` | `2 / 0.70` | Limits instruction-bearing matches and rejects weak semantic matches. |
| `ToolResultPolicy` | `offload_above_chars` | `320` | Makes the long diagnostic result a durable pointer. |
| `ToolResultPolicy` | `expansion_tool_names` | three safe names | Prevents pointer loops and keeps KB evidence inline. |


In [17]:
RETRIEVAL_POLICY = RetrievalPolicy(
    conversation_scope="thread",
    knowledge_base_scope="namespace",
    knowledge_base_namespaces=(KB_NAMESPACE,),
    candidate_limit=6,
    max_items=5,
    # Both the SLO and severity paragraphs are needed from one source.
    dedupe_parent_sources=False,
)

SKILL_RETRIEVAL_CONFIG = {
    "top_k": 2,
    "min_similarity": 0.70,
}

TOOL_RESULT_POLICY = ToolResultPolicy(
    offload_above_chars=320,
    expansion_tool_names=frozenset(
        {
            "retrieve_tool_log_entry",
            "expand_tool_result",
            "knowledge_base_lookup",
        }
    ),
)


### Context types and Nova's evidence contract

`MEMAGENT` is durable configuration, not prompt context. The instruction forces Nova to state when a required representation is absent and keeps live state behind a trusted tool.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemoryType` selection | `exclude` | `MEMAGENT` | Avoids treating saved configuration as retrievable evidence. |
| system instruction | `evidence source` | supplied memory or trusted tool | Prevents general-knowledge guessing. |
| system instruction | `live state` | `service_health` only | Separates durable facts from current telemetry. |
| system instruction | `stopping rule` | stop when sufficient | Reduces redundant lookups and tool loops. |

In [18]:
CONTEXT_MEMORY_TYPES = [
    memory_type for memory_type in MemoryType if memory_type != MemoryType.MEMAGENT
]

NOVA_INSTRUCTION = """You are Nova, a production-support agent.

Outcome: answer the user's support question concisely and make the evidence boundary visible.

Evidence contract:
- Use facts only when they appear in this turn's supplied MemoRizz memory/context or a trusted tool result.
- If required evidence is absent, say that it is unavailable in supplied memory; do not fill gaps from general knowledge.
- Treat a procedure as reviewed only when a Skill Memory section supplies it.
- For current or live service health, call service_health when that tool is available; otherwise say live health cannot be checked.
- Never infer current telemetry from conversation, entity, knowledge, workflow, or skill memory.
- Honor retrieved user formatting and escalation preferences.
- If a memory lookup returns no matches, do not retry it with rephrased queries in the same turn; state that evidence is unavailable.
- Stop calling tools as soon as the supplied result is sufficient to answer the request.
- When the user asks only for a procedure and explicitly says not to execute it, summarize applicable reviewed Skill Memory without calling the procedure's tools.
"""


### Create the baseline Nova agent

All context-capable types are enabled, but their stores are empty. Progressive
disclosure and entity inference are held back so later experiments change one
variable at a time. Skill retrieval is on, but an empty Skillbox contributes no
instructions.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent` | `memory_ids` | `[MEMORY_ID]` | Binds Nova to this incident's content scope. |
| `MemAgent` | `memory_types` | all except `MEMAGENT` | Enables the connected lifecycle without putting config in the prompt. |
| `MemAgent` | `max_steps` | `8` | Bounds tool-loop iterations. |
| `ContextPolicy` | `progressive_tool_disclosure` | `False` | Keeps discovery tools out of baseline comparisons. |
| `MemAgent` | `skill_retrieval` | `True` | Creates an initially empty Skillbox and enables later semantic use. |
| `MemAgent` | `learning_control_plane / automations` | `False / False` | Keeps background behavior and side effects out of the lesson. |


In [19]:
teaching_model = make_model()

support_agent = MemAgent(
    model=teaching_model,
    name=f"Nova Support {RUN_ID}",
    application_id=f"memory-types-course-{RUN_ID}",
    instruction=NOVA_INSTRUCTION,
    memory_provider=provider,
    memory_ids=[MEMORY_ID],
    memory_types=CONTEXT_MEMORY_TYPES,
    max_steps=8,
    context_policy=ContextPolicy(
        progressive_tool_disclosure=False,
        tool_top_k=4,
    ),
    retrieval_policy=RETRIEVAL_POLICY,
    tool_result_policy=TOOL_RESULT_POLICY,
    skill_retrieval=True,
    skill_retrieval_config=SKILL_RETRIEVAL_CONFIG,
    learning_control_plane=False,
    automations_enabled=False,
)


### Persist Nova's identity and hold Entity memory back

Saving now creates the operational `MEMAGENT` record. Entity tools stay hidden
until their dedicated section so they cannot affect earlier baselines.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `support_agent.save` | `—` | baseline configuration | Creates durable identity before memory is populated. |
| `with_entity_memory` | `enabled` | `False` | Isolates the later Entity-memory experiment. |


In [20]:
support_agent.save()
AGENT_ID = support_agent.agent_id

# Keep the next layer genuinely absent during the episodic/persona baseline.
# The enum remains configured on the durable agent, but entity retrieval and
# its model-callable tools are enabled only when Section 5 introduces them.
support_agent.with_entity_memory(False)


### Inspect the baseline configuration

This report confirms identity, active context types, and the intentionally
hidden entity lookup before the incident experiments begin.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `active_memory_types` | `—` | runtime list | Shows which representations can participate. |
| `tool_manager.list_tools` | `—` | current schemas | Proves entity inference is not an accidental baseline variable. |


In [21]:
print("Agent:", {"agent_id": AGENT_ID, "memory_id": MEMORY_ID})
print(
    "Context-relevant active types:",
    [item.value for item in support_agent.active_memory_types],
)
print("Operational type exercised by save/load:", MemoryType.MEMAGENT.value)
print(
    "Entity inference tool visible:",
    "entity_memory_lookup" in support_agent.tool_manager.list_tools(),
)


Agent: {'agent_id': 'e07af414-5d39-5ea1-805c-4a8d0a9c82cf', 'memory_id': 'nova-support-0b6953f061'}
Context-relevant active types: ['personas', 'toolbox', 'entity_memory', 'short_term_memory', 'knowledge_base', 'conversation_memory', 'workflow_memory', 'skillbox', 'shared_memory', 'summaries', 'semantic_cache', 'tool_log']
Operational type exercised by save/load: agents
Entity inference tool visible: False


# Part I · Episodic memory

## 2 · `CONVERSATION_MEMORY`: what happened in this thread?

A raw model call has no continuity unless earlier messages are supplied again.
`MemAgent.run()` loads an exact scoped history, builds bounded context, calls the
model, then writes the user and assistant turns. The first question below must
fail because Maya has not stated a preference. After she states it, the same
question succeeds. A different `user_id` still fails, proving that shared
`memory_id` and `thread_id` strings do not defeat tenant isolation.

**Unit shape:** role, content, timestamp, memory/user/thread/agent identifiers,
provenance, embedding, and optional `summary_id`.

A tool-using turn can also add `tool`-role rows. For that reason the audit counts
the three expected user/assistant pairs and reports any extra tool rows instead
of assuming that every run always writes exactly two physical records.

**Do not use it for:** canonical service ownership or current telemetry. A past
utterance is evidence that something was said, not proof that it is still true.

### Run the conversation before/after/isolation experiment

Every `run()` prints a user-visible answer. The same question is asked before memory, after Maya's preference, and as another user.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent.run` | `memory_id` | `MEMORY_ID` | Selects the incident memory scope. |
| `MemAgent.run` | `user_id` | Maya then another user | Tests tenant isolation. |
| `MemAgent.run` | `thread_id` | `THREAD_ID` | Keeps the dialogue in one ordered thread. |
| `MemAgent.run` | `message` | question or preference | Writes the turn and returns Nova's answer. |

In [23]:
scope = {"memory_id": MEMORY_ID, "user_id": USER_ID, "thread_id": THREAD_ID}
preference_question = (
    "What do you remember about my incident-update format and escalation preference?"
)

before_conversation = support_agent.run(preference_question, **scope)
before_conversation_prompt = teaching_model.prompts[-1]["text"].lower()

acknowledgement = support_agent.run(
    "Remember this for incidents: keep every incident update under five bullets. "
    "Page me before emailing executives.",
    **scope,
)
after_conversation = support_agent.run(preference_question, **scope)
after_conversation_prompt = teaching_model.prompts[-1]["text"].lower()
isolated_user = support_agent.run(
    preference_question,
    memory_id=MEMORY_ID,
    user_id=OTHER_USER_ID,
    thread_id=THREAD_ID,
)
isolated_user_prompt = teaching_model.prompts[-1]["text"].lower()

print("Before memory:", before_conversation)
print("Preference acknowledgement:", acknowledgement)
print("After memory:", after_conversation)
print("Different user:", isolated_user)


Before memory: I don’t have any supplied memory in this thread that states your incident-update format or escalation preference.

Evidence boundary: I checked the available conversation summaries registry and found no summaries; the current conversation also doesn’t include those preferences.
Preference acknowledgement: Got it. For incidents, I’ll follow these preferences:

- Keep every incident update under five bullets.
- Page you before emailing executives.
After memory: I remember:

- Incident updates should be under five bullets.
- Page you before emailing executives.
Different user: I don’t have any supplied memory in this thread about your incident-update format or escalation preference.

Evidence boundary: I checked available summaries and recent tool logs; both are empty. No current conversation context includes those preferences.


### Prove conversation-memory causality

Prompt inspection proves the preference entered the model only after the stored turn and remained absent for the other user.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| prompt audit | `before` | preference absent | Establishes the baseline. |
| prompt audit | `after` | both preferences present | Proves memory utilization rather than lucky wording. |
| prompt audit | `other user` | preference absent | Proves user scoping. |
| `assert` | `all effects` | `True` | Turns the narrative into executable evidence. |

In [24]:
conversation_effect = {
    "preference_absent_before": "under five bullets" not in before_conversation_prompt,
    "preference_injected_after": "under five bullets" in after_conversation_prompt
    and "before emailing executives" in after_conversation_prompt,
    "different_user_isolated": "under five bullets" not in isolated_user_prompt,
}

display(
    Markdown(
        "| Experiment | Nova's answer |\n|---|---|\n"
        f"| Before a stored turn | {before_conversation} |\n"
        f"| After Maya states it | {after_conversation} |\n"
        f"| Same IDs, different user | {isolated_user} |"
    )
)
print("Programmatic effect:", conversation_effect)
assert all(conversation_effect.values())
assert all(isinstance(answer, str) and answer.strip() for answer in (
    before_conversation, acknowledgement, after_conversation, isolated_user
))


| Experiment | Nova's answer |
|---|---|
| Before a stored turn | I don’t have any supplied memory in this thread that states your incident-update format or escalation preference.

Evidence boundary: I checked the available conversation summaries registry and found no summaries; the current conversation also doesn’t include those preferences. |
| After Maya states it | I remember:

- Incident updates should be under five bullets.
- Page you before emailing executives. |
| Same IDs, different user | I don’t have any supplied memory in this thread about your incident-update format or escalation preference.

Evidence boundary: I checked available summaries and recent tool logs; both are empty. No current conversation context includes those preferences. |

Programmatic effect: {'preference_absent_before': True, 'preference_injected_after': True, 'different_user_isolated': True}


### Inspect the physical conversation rows

The provider read is exact and ordered; it is used for audit rather than semantic recall.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `retrieve_conversation_history_ordered_by_timestamp` | `memory_id / user_id / thread_id` | exact scope | Prevents accidental administrative reads. |
| provider read | `memory_type` | `CONVERSATION_MEMORY` | Selects the episodic store. |
| role filter | `user and assistant` | three pairs | Distinguishes dialogue from tool-role audit rows. |

In [25]:
conversation_rows = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)


compact_history = [
    {
        "role": row.get("role"),
        "content": row.get("content"),
        "user_id": row.get("user_id"),
        "thread_id": row.get("thread_id") or row.get("conversation_id"),
    }
    for row in conversation_rows
]

dialogue_rows = [
    row for row in compact_history if row["role"] in {"user", "assistant"}
]

role_counts = {
    role: sum(row["role"] == role for row in compact_history)
    for role in sorted({row["role"] for row in compact_history})
}

display(compact_history)
print("Role counts:", role_counts)
assert len(dialogue_rows) == 6
assert [row["role"] for row in dialogue_rows] == ["user", "assistant"] * 3
assert all(row["user_id"] == USER_ID for row in compact_history)
assert all(row["thread_id"] == THREAD_ID for row in compact_history)


[{'role': 'user',
  'content': 'What do you remember about my incident-update format and escalation preference?',
  'user_id': 'maya-0b6953f061',
  'thread_id': 'checkout-incident'},
 {'role': 'assistant',
  'content': 'I don’t have any supplied memory in this thread that states your incident-update format or escalation preference.\n\nEvidence boundary: I checked the available conversation summaries registry and found no summaries; the current conversation also doesn’t include those preferences.',
  'user_id': 'maya-0b6953f061',
  'thread_id': 'checkout-incident'},
 {'role': 'user',
  'content': 'Remember this for incidents: keep every incident update under five bullets. Page me before emailing executives.',
  'user_id': 'maya-0b6953f061',
  'thread_id': 'checkout-incident'},
 {'role': 'assistant',
  'content': 'Got it. For incidents, I’ll follow these preferences:\n\n- Keep every incident update under five bullets.\n- Page you before emailing executives.',
  'user_id': 'maya-0b6953f06

Role counts: {'assistant': 3, 'user': 3}


## 3 · `SUMMARIES`: compress history without severing provenance

Long transcripts are durable but expensive to replay. `generate_summaries()`
groups unsummarized messages inside one memory/user/thread boundary, asks the
model for a compact representation, stores the summary, and links every source
message. The original rows receive a `summary_id`; they are not deleted.

The experiment proves three different properties:

1. the summary count equals the number of linked source IDs;
2. those IDs expand back to the original role/content rows;
3. Nova still recalls the preference after normal history loading omits the
   summarized turns and surfaces the compact summary reference instead.

Summarization is a **context policy**, not a retention policy. Legal deletion,
user erasure, and TTL cleanup must use explicit lifecycle operations.

### Generate and expand a source-linked summary

Summarization compacts prompt history while retaining exact links to every source row.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `generate_summaries` | `days_back` | `7` | Limits which unsummarized turns are eligible. |
| `generate_summaries` | `max_memories_per_summary` | `20` | Bounds one summary unit. |
| `fetch_context_summary` | `summary_id` | new summary | Retrieves the compact representation in the same scope. |
| `get_messages_by_ids` | `source_message_ids` | linked IDs | Expands back to original evidence. |

In [26]:
import pprint
summary_ids = support_agent.generate_summaries(
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    days_back=7,
    max_memories_per_summary=20,
)

assert len(summary_ids) == 1

summary = support_agent.fetch_context_summary(
    summary_ids[0],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)

expanded = support_agent.memory_manager.get_messages_by_ids(
    summary["source_message_ids"],
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
pprint.pprint(
    {
        "summary": summary["content"],
        "memory_units_count": summary["memory_units_count"],
        "source_id_count": len(summary["source_message_ids"]),
        "expanded_roles": [row.get("role") for row in expanded],
    }
)
assert summary["memory_units_count"] == len(summary["source_message_ids"])
assert len(expanded) == summary["memory_units_count"]


{'expanded_roles': ['user',
                    'assistant',
                    'user',
                    'assistant',
                    'user',
                    'assistant'],
 'memory_units_count': 6,
 'source_id_count': 6,
 'summary': 'The user established and confirmed incident communication '
            'preferences:\n'
            '\n'
            '- Incident updates must be concise: under five bullets.\n'
            '- Escalation order matters: page the user before emailing '
            'executives.\n'
            '- The assistant initially had no prior memory of these '
            'preferences, then stored and correctly recalled them.\n'
            '- Key pattern: the user values brief operational updates and '
            'wants direct escalation before executive notification.'}


### Run Nova after compaction

The agent answers again from the summary path; exact row lookups verify that source messages carry the summary marker.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent.run` | `message` | preference question | Shows user-visible recall after compaction. |
| `retrieve_by_id` | `memory type` | `CONVERSATION_MEMORY` | Verifies each source row's `summary_id`. |
| prompt audit | `expected phrase` | `under five bullets` | Proves the compact representation reached GPT. |

In [27]:
recalled_after_compaction = support_agent.run(preference_question, **scope)

compacted_recall_prompt = teaching_model.prompts[-1]["text"].lower()

marked_sources = [
    provider.retrieve_by_id(source_id, MemoryType.CONVERSATION_MEMORY)
    for source_id in summary["source_message_ids"]
]

print("Recall after compaction:", recalled_after_compaction)
print(
    "Source links preserved:",
    all(row and str(row.get("summary_id")) == str(summary_ids[0]) for row in marked_sources),
)
assert "under five bullets" in compacted_recall_prompt
assert isinstance(recalled_after_compaction, str) and recalled_after_compaction.strip()


Recall after compaction: I remember:

- Incident updates should be under five bullets.
- Page you before emailing executives.

Evidence boundary: this comes from supplied memory in this turn.
Source links preserved: True


# Part II · Semantic memory

## 4 · `PERSONAS`: who is the agent across incidents?

A persona belongs to Nova, not Maya. It holds stable identity fields—name, role,
goals, and background—and evolves through versioned changes with a reason and
source pointer. Without one, Nova has only its generic instruction. Attaching a
persona changes its operating stance on every thread; evolving it adds an
auditable requirement to state uncertainty.

Use persona evolution for durable identity shifts. Do not put customer
preferences or service facts here: those belong in conversation/entity memory
and have different owners and correction rules.

### Compare no persona with an attached persona

A Persona is agent-owned identity, so each answer uses a separate thread while keeping the same agent and user.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `Persona` | `name / role` | Nova / customer support | Defines stable identity. |
| `Persona` | `goals / background` | incident stance | Provides the behavior expected in every thread. |
| `set_persona` | `save` | `True` | Attaches the persona and stores its provider projection. |
| `MemAgent.run` | `thread_id` | before then attached | Makes the before/after prompt boundary visible. |

In [ ]:
stance_question = "What operating stance will you take during production incidents?"
before_persona = support_agent.run(
    stance_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="persona-before",
)
before_persona_prompt = teaching_model.prompts[-1]["text"].lower()



In [29]:
print("Without persona:", before_persona)


Without persona: During production incidents, I’ll take a **production-support stance**:

- **Evidence-bound:** I’ll only state facts supported by the current supplied context/memory or trusted tool results.
- **No guessing:** If evidence is missing, I’ll say it’s unavailable rather than infer.
- **Live health discipline:** For current service health, I’ll use a `service_health` tool if available; otherwise I’ll state that live health cannot be checked.
- **Procedure safety:** I’ll treat procedures as reviewed only if supplied via Skill Memory, and I won’t execute procedures when you ask only for a summary.
- **Concise incident communication:** I’ll focus on impact, evidence, next steps, and clear escalation boundaries.
- **Stop when sufficient:** I’ll avoid unnecessary tool calls once enough evidence exists to answer.


In [30]:

persona = Persona(
    name="Nova",
    role=RoleType.CUSTOMER_SUPPORT,
    goals="Stay calm, evidence-first, and concise during production incidents.",
    background="A production-support specialist for payment services.",
)

assert support_agent.set_persona(persona, save=True)

PERSONA_STORAGE_ID = persona._storage_id


In [31]:

with_persona = support_agent.run(
    stance_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="persona-attached",
)
with_persona_prompt = teaching_model.prompts[-1]["text"].lower()

print("With persona:", with_persona)


With persona: During production incidents, I’ll take a calm, evidence-first support stance:

- Prioritize current supplied context and trusted tool results.
- For live/current service health, use the appropriate health-check tool if available; otherwise state that live health can’t be checked.
- Avoid inferring telemetry or incident state from memory alone.
- Make the evidence boundary explicit: what is known, what is unavailable, and what needs escalation or verification.
- Keep guidance concise, actionable, and customer-focused.


### Evolve Persona v1 into reviewed v2

Evolution changes a durable identity field and attaches provenance explaining who requested the change and why.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `Persona.update` | `updates` | new `goals` | Adds an explicit uncertainty rule. |
| `Persona.update` | `change_trigger.reason` | review requirement | Makes the governance decision auditable. |
| `Persona.update` | `source_type / source_id` | user feedback / review ID | Links the change to its evidence. |
| `Persona.update` | `provider` | Oracle | Persists the provider-supported projection. |

In [ ]:
evolution = persona.update(
    updates={
        "goals": (
            "Stay calm, evidence-first, and concise during production incidents; "
            "state uncertainty explicitly and never fabricate live state."
        )
    },
    change_trigger={
        "reason": "The incident-response review required explicit uncertainty.",
        "source_type": "user_feedback",
        "source_id": f"incident-review-{RUN_ID}",
        "agent_id": AGENT_ID,
    },
    provider=provider,
)


After persona evolution: During production incidents, I’ll take an evidence-first support stance:

- Stay calm, concise, and customer-focused.
- Use only supplied memory/context or trusted tool results as evidence.
- State uncertainty explicitly when evidence is missing.
- Never fabricate or infer live service state.
- For current service health, use a live health tool if available; otherwise say live health cannot be checked.
- Escalate clearly when the evidence indicates risk or when required data is unavailable.


In [ ]:

# Keep the reviewer-approved rich snapshot at the application boundary. The
# Oracle relational projection caveat is demonstrated during MEMAGENT reload.
reviewed_persona_snapshot = dict(evolution["persona"])
# Refresh the live attachment. Oracle persists its stable base projection;
# the reviewed rich snapshot above remains the source for versioned rebind.
assert support_agent.set_persona(persona, save=True)
PERSONA_STORAGE_ID = persona._storage_id
after_evolution = support_agent.run(
    stance_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="persona-evolved",
)
evolved_persona_prompt = teaching_model.prompts[-1]["text"].lower()

print("After persona evolution:", after_evolution)


### Verify the Persona effect

The answer is printed above; this block checks the prompts and version history so a prose change alone cannot pass the lesson.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| prompt audit | `before / attached / evolved` | three states | Isolates the effect of Persona memory. |
| `evolution_history` | `latest trigger` | review metadata | Shows why v2 exists. |
| `assert` | `version` | `2` | Verifies a real version transition. |

In [21]:
persona_effect = {
    "persona_absent_before": (
        "production-support specialist for payment services"
        not in before_persona_prompt
    ),
    "persona_injected_after_attach": (
        "stay calm, evidence-first" in with_persona_prompt
    ),
    "versioned_goal_injected": (
        "state uncertainty explicitly" in evolved_persona_prompt
    ),
}

persona_answers = [
    ("Before Persona memory", before_persona),
    ("After Persona attachment", with_persona),
    (f"After Persona evolution (version {persona.version})", after_evolution),
]
for label, answer in persona_answers:
    display(Markdown(f"#### {label}\n\n{answer}"))

print("Latest evolution trigger:", persona.evolution_history[-1]["change_trigger"])
print("Programmatic effect:", persona_effect)
assert persona.version == 2 and evolution["updated"]
assert all(persona_effect.values())


#### Before Persona memory

During production incidents, I’ll take a **production-support stance**:

- **Evidence-bound:** I’ll only state facts that are present in the supplied memory/context or trusted tool results.
- **No guessing:** If evidence is missing, I’ll say it’s unavailable rather than infer or fill gaps.
- **Live health boundary:** For current service health, I’ll call the appropriate live health tool if available; if not, I’ll state that live health cannot be checked.
- **Procedure discipline:** I’ll treat a procedure as reviewed only if it comes from supplied Skill Memory.
- **Telemetry caution:** I won’t infer current telemetry from prior conversations, memory, workflows, or skills.
- **Concise incident support:** I’ll focus on clear status, evidence, next steps, and escalation boundaries.

#### After Persona attachment

During production incidents, I’ll stay calm, evidence-first, and concise.

My stance:
- Prioritize live/trusted evidence over assumptions.
- Make the evidence boundary clear: if supplied memory or trusted tools don’t show something, I’ll say it’s unavailable.
- Avoid inferring current service health from stale memory.
- Follow reviewed procedures only when supplied via Skill Memory.
- Escalate clearly when evidence is insufficient or impact is production-facing.
- Focus on prompt customer resolution with clear next steps.

#### After Persona evolution (version 2)

During production incidents, I’ll take an evidence-first support stance:

- Stay calm, concise, and customer-focused.
- Use only supplied memory/context or trusted tool results as evidence.
- State uncertainty explicitly when evidence is missing.
- Never fabricate or infer live service state.
- For current service health, use a live health tool if available; otherwise say live health cannot be checked.
- Escalate clearly when the evidence indicates risk or when required data is unavailable.

Latest evolution trigger: {'reason': 'The incident-response review required explicit uncertainty.', 'source_type': 'user_feedback', 'source_id': 'incident-review-30e75d0141', 'conversation_id': None, 'agent_id': 'a62f6cee-d175-5b38-b969-510df5ae1852', 'triggered_at': '2026-08-25T14:55:08.111227'}
Programmatic effect: {'persona_absent_before': True, 'persona_injected_after_attach': True, 'versioned_goal_injected': True}


## 5 · `ENTITY_MEMORY`: structured, correctable facts about the service

Conversation recall could find a sentence saying “Maya owns checkout-api,” but
applications usually need a canonical profile with attribute-level provenance
and confidence. Entity memory stores exactly that, plus typed relations.

The first answer fails. We then upsert the service and add the escalation
channel from a second source. Automatic entity retrieval injects a simplified
profile into the next turn. An exact lookup under another tenant returns no
record, demonstrating that entity IDs are not global authorization tokens.

Time-sensitive telemetry still does not belong here. “Deployment is currently
unhealthy” should come from a live tool; persisting it as a durable entity fact
would create stale confidence.

### Establish the entity baseline and write canonical facts

Entity attributes are independently sourced and correctable; volatile live health is intentionally excluded.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `upsert_entity` | `entity_id / name / entity_type` | checkout service | Creates one canonical object. |
| `upsert_entity` | `attributes` | owner, tier, region | Stores typed facts with confidence and provenance. |
| `upsert_entity` | `relations` | owned-by relation | Connects the service to another entity. |
| `record_attribute` | `attribute_name / value` | escalation channel | Adds a separately sourced fact. |
| entity writes | `memory_id / user_id` | current scope | Enforces tenant isolation. |

In [22]:
ownership_question = "Who owns checkout-api and which channel should I use to escalate?"
before_entity = support_agent.run(
    ownership_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="entity-before",
)
before_entity_prompt = teaching_model.prompts[-1]["text"].lower()

entities = EntityMemory(provider)
entities.upsert_entity(
    entity_id=ENTITY_ID,
    name="checkout-api",
    entity_type="service",
    attributes=[
        {"name": "owner", "value": "Maya Chen", "confidence": 0.99, "source": "service-catalog"},
        {"name": "tier", "value": "tier-1", "confidence": 0.98, "source": "service-catalog"},
        {"name": "region", "value": "eu-west", "confidence": 0.96, "source": "deployment-registry"},
    ],
    relations=[
        {"entity_id": f"maya-chen-{RUN_ID}", "relation_type": "owned_by", "confidence": 0.99}
    ],
    metadata={"catalog_version": "2026-08-25"},
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
entities.record_attribute(
    entity_id=ENTITY_ID,
    attribute_name="escalation_channel",
    attribute_value="#checkout-incidents",
    confidence=0.97,
    source="on-call-directory",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)


'checkout-api-30e75d0141'

### Enable scoped entity recall and inspect its evidence

Entity inference is enabled only after the baseline, then disabled again so the next knowledge experiment changes one memory layer.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `with_entity_memory` | `enabled` | `True` then `False` | Controls automatic entity retrieval/tools at the experiment boundary. |
| `search_entities_with_diagnostics` | `query` | owner escalation | Shows real semantic retrieval mode and match count. |
| `get_entity` | `OTHER_USER_ID` | different tenant | Must return no record. |
| `MemAgent.run` | `thread_id` | `entity-after` | Prints the user-visible answer with injected facts. |

In [23]:
# Activate automatic scoped entity retrieval only after the canonical profile
# exists. This makes the before/after boundary visible to both GPT and the audit.
support_agent.with_entity_memory(True)

after_entity = support_agent.run(
    ownership_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="entity-after",
)
after_entity_prompt = teaching_model.prompts[-1]["text"].lower()
record = entities.get_entity(ENTITY_ID, memory_id=MEMORY_ID, user_id=USER_ID)
matches, diagnostics = entities.search_entities_with_diagnostics(
    "checkout service owner escalation",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    limit=3,
)
other_tenant_record = entities.get_entity(
    ENTITY_ID, memory_id=MEMORY_ID, user_id=OTHER_USER_ID
)

print("Before:", before_entity)
print("After :", after_entity)
print(
    "Stored attributes:",
    [(item["name"], item["value"], item["source"]) for item in record["attributes"]],
)
print("Retrieval diagnostics:", {k: diagnostics.get(k) for k in ("retrieval_mode", "fallback_used", "match_count")})
entity_effect = {
    "owner_absent_before": "maya chen" not in before_entity_prompt,
    "owner_injected_after": "maya chen" in after_entity_prompt,
    "channel_injected_after": "#checkout-incidents" in after_entity_prompt,
}
print("Programmatic effect:", entity_effect)
assert all(entity_effect.values())
assert matches and other_tenant_record is None

# The entity effect is now proven. Pause its model-callable lookup during the
# next baseline so the knowledge experiment changes one layer at a time.
support_agent.with_entity_memory(False)


Before: I don’t have ownership or escalation-channel evidence for `checkout-api` in the supplied memory.

Evidence checked: knowledge base lookup for `checkout-api owner escalation channel` returned no attached entries/matches.
After : checkout-api is owned by Maya Chen. Escalate via `#checkout-incidents`.

Evidence: supplied entity memory in this turn.
Stored attributes: [('owner', 'Maya Chen', 'service-catalog'), ('tier', 'tier-1', 'service-catalog'), ('region', 'eu-west', 'deployment-registry'), ('escalation_channel', '#checkout-incidents', 'on-call-directory')]
Retrieval diagnostics: {'retrieval_mode': 'semantic', 'fallback_used': False, 'match_count': 1}
Programmatic effect: {'owner_absent_before': True, 'owner_injected_after': True, 'channel_injected_after': True}


## 6 · `KNOWLEDGE_BASE`: source passages, chunking, and scoped retrieval

Entity memory gives Nova an owner and channel, but it still does not know the
approved SLO or SEV-1 policy. Those facts live in a runbook whose passages need
provenance and semantic retrieval.

`ingest_knowledge()` assigns one `knowledge_base_id`, splits the source, embeds
each chunk, and stores chunk index/count/strategy plus namespace and tenant.
Paragraph chunking fits this short structured runbook. Other supported choices
are `fixed`, `sentence`, `semantic`, `none`, or a custom callable. For real files,
`ingest_file()` and `ingest_directory()` use the shared extractor registry.

Attaching the knowledge base records the relationship on Nova and constrains the
model-callable `knowledge_base_lookup` to attached document IDs. The explicit
provider query is independently restricted by namespace and tenant. Attachment
is therefore discoverability—not a replacement for retrieval scope.

This lesson sets `dedupe_parent_sources=False` because one question needs two
complementary paragraphs from the same short runbook: the 300 ms objective and
the 450 ms / 10-minute severity gate. Parent deduplication is valuable when
sibling chunks are repetitive, but applying it blindly here would discard
necessary evidence. The before/after assertion checks for both passages.

A 240-character generic offload threshold would compact this multi-chunk lookup.
We therefore list `knowledge_base_lookup` in the policy's expansion-tool set so
its evidence remains inline, while the later long `service_health` result still
becomes an auditable pointer. This shows why retrieval and tool-result policies
must be tuned together.

### Establish the knowledge baseline and ingest the runbook

Paragraph chunking deliberately creates separate SLO, SEV, rollback, and escalation passages.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `KnowledgeBase.ingest_knowledge` | `namespace` | `KB_NAMESPACE` | Creates a retrieval boundary for this runbook. |
| `KnowledgeBase.ingest_knowledge` | `chunking_strategy` | `paragraph` | Preserves the policy's logical sections. |
| `KnowledgeBase.ingest_knowledge` | `user_id` | Maya scope | Prevents cross-tenant retrieval. |
| `attach_to_agent` | `knowledge_base_id` | new KB ID | Makes the corpus eligible for Nova. |

In [24]:
policy_question = "What is checkout-api's latency objective, and when must we declare SEV-1?"
before_knowledge = support_agent.run(
    policy_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="knowledge-before",
)
before_knowledge_prompt = teaching_model.prompts[-1]["text"].lower()

runbook = """Checkout API service objective: production p95 latency must remain below 300 milliseconds, and the error rate must remain below 1 percent.

Declare SEV-1 when p95 latency exceeds 450 milliseconds for 10 continuous minutes or when errors exceed 5 percent. Every SEV-1 requires an incident commander and a communications lead.

If a new deployment causes sustained error-budget burn, prefer rollback after checking live health and confirming the previous release is safe.

During escalation, use the service-catalog owner and the on-call directory. Do not infer current health from this document."""

knowledge = KnowledgeBase(provider)
KB_ID = knowledge.ingest_knowledge(
    runbook,
    namespace=KB_NAMESPACE,
    chunking_strategy="paragraph",
    user_id=USER_ID,
)
assert knowledge.attach_to_agent(support_agent, KB_ID)


### Retrieve policy evidence and run Nova again

A direct provider query exposes the raw top passage while `MemAgent.run()` shows the grounded user-visible result.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `retrieve_by_query` | `query` | latency severity threshold | Embeds a semantic search request. |
| `retrieve_by_query` | `namespace / user_id` | exact run scope | Applies knowledge isolation. |
| `retrieve_by_query` | `limit` | `3` | Bounds raw evidence returned for inspection. |
| `MemAgent.run` | `thread_id` | `knowledge-after` | Prints the answer after KB attachment. |

In [25]:
direct_hits = provider.retrieve_by_query(
    "checkout latency severity threshold",
    memory_store_type=MemoryType.KNOWLEDGE_BASE,
    namespace=KB_NAMESPACE,
    user_id=USER_ID,
    limit=3,
)
after_knowledge = support_agent.run(
    policy_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="knowledge-after",
)
after_knowledge_prompt = teaching_model.prompts[-1]["text"].lower()

chunks = knowledge.retrieve_knowledge(KB_ID)
print("Before:", before_knowledge)
print("After :", after_knowledge)
print(
    "Chunks:",
    [
        {
            "index": row.get("chunk_index"),
            "count": row.get("chunk_count"),
            "strategy": row.get("chunking_strategy"),
            "preview": row.get("content", "")[:80],
        }
        for row in chunks
    ],
)
print("Top direct hit:", direct_hits[0]["content"])
knowledge_effect = {
    "threshold_absent_before": "450 milliseconds" not in before_knowledge_prompt,
    "slo_injected_after": "below 300 milliseconds" in after_knowledge_prompt,
    "severity_gate_injected_after": "10 continuous minutes" in after_knowledge_prompt,
    "answer_uses_both_thresholds": "300" in after_knowledge and "450" in after_knowledge,
}
print("Programmatic effect:", knowledge_effect)
assert len(chunks) == 4
assert direct_hits and all(knowledge_effect.values())


Before: I don’t have the needed evidence in supplied memory.

- **checkout-api latency objective:** unavailable in supplied memory.
- **When to declare SEV-1:** unavailable in supplied memory.

No knowledge base entries are attached, and no reviewed Skill Memory/procedure was supplied for this topic.
After : checkout-api’s latency objective: production p95 latency must stay below **300 ms**.

Declare **SEV-1** when either:
- p95 latency exceeds **450 ms for 10 continuous minutes**, or
- errors exceed **5%**.

Evidence: retrieved from `checkout-runbook-30e75d0141`.
Chunks: [{'index': 0, 'count': 4, 'strategy': 'paragraph', 'preview': 'Checkout API service objective: production p95 latency must remain below 300 mil'}, {'index': 1, 'count': 4, 'strategy': 'paragraph', 'preview': 'Declare SEV-1 when p95 latency exceeds 450 milliseconds for 10 continuous minute'}, {'index': 2, 'count': 4, 'strategy': 'paragraph', 'preview': 'If a new deployment causes sustained error-budget burn, prefer rol

# Part III · Procedural and operational memory

## 7 · `TOOLBOX`: capability metadata plus a trusted binding

Durable facts still cannot answer “what is the live state?” Without a tool,
Nova correctly refuses. The host then registers a read-only deterministic
callable. `Toolbox.from_functions(..., augment=False)` derives a strict JSON
Schema without an LLM, persists only metadata/policy, and keeps the executable
Python binding in the trusted process.

Tool schema describes capability; it does not grant authority. The
`@governed_tool` policy declares that this function is deterministic,
side-effect free, and belongs to the `service-health` invalidation domain.
Mutations would need an approval policy and should normally bypass cache.

We reconstruct the same persisted Nova identity with the new toolbox. This is a
capability upgrade, not a new agent: conversation, summaries, entity facts,
persona, and knowledge stay connected through the same agent/memory scope. This
reconstruction also turns on progressive disclosure; the router preview proves
that only the relevant live-health schema is selected from the broader tool set.

### Show the missing capability, then register a governed tool

The baseline proves memory cannot fabricate live state. The decorator declares operational characteristics; it does not itself run the function.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `governed_tool` | `deterministic` | `True` | Same input returns the same teaching fixture. |
| `governed_tool` | `side_effects` | `False` | The read-only check needs no mutation approval. |
| `governed_tool` | `domains` | `service-health` | Supports domain-aware cache invalidation/governance. |
| `Toolbox.from_functions` | `augment` | `False` | Derives schema without an LLM augmentation call. |
| `Toolbox.from_functions` | `agent_id / user_id` | current scope | Persists metadata under the correct owner. |

In [26]:
live_question = "What is checkout-api's live state right now?"
before_tool = support_agent.run(
    live_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="tool-before",
)
before_tool_schemas = teaching_model.prompts[-1]["tool_names"]

@governed_tool(deterministic=True, side_effects=False, domains=("service-health",))
def service_health(service: str) -> dict:
    """Read live service health, latency, errors, deployment, and bounded diagnostics."""
    return {
        "service": service,
        "status": "degraded",
        "p95_ms": 480,
        "error_rate_pct": 2.2,
        "deployment": "blue-2026-08-25",
        "diagnostics": [f"probe-{index:02d}: latency elevated" for index in range(24)],
    }


toolbox = Toolbox.from_functions(
    [service_health],
    memory_provider=provider,
    agent_id=AGENT_ID,
    user_id=USER_ID,
    augment=False,
)
tool_metadata = toolbox.get_tool_by_name("service_health")


### Reconstruct Nova with the trusted Toolbox binding

The durable agent ID and memories stay constant; only capability binding and progressive disclosure change.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent` | `agent_id` | existing `AGENT_ID` | Preserves identity and workflow ownership. |
| `MemAgent` | `toolbox` | persisted metadata plus callable | Makes `service_health` executable in this process. |
| `ContextPolicy` | `progressive_tool_disclosure` | `True` | Lets semantic routing expose only relevant tool schemas. |
| `attach_to_agent` | `KB_ID` | existing runbook | Restores the connected knowledge attachment after reconstruction. |

In [27]:
# Recreate the runtime with the same durable identity plus the trusted binding.
support_agent.close(close_memory_provider=False)
support_agent = MemAgent(
    model=teaching_model,
    name=f"Nova Support {RUN_ID}",
    application_id=f"memory-types-course-{RUN_ID}",
    agent_id=AGENT_ID,
    instruction=NOVA_INSTRUCTION,
    persona=persona,
    toolbox=toolbox,
    memory_provider=provider,
    memory_ids=[MEMORY_ID],
    memory_types=CONTEXT_MEMORY_TYPES,
    max_steps=8,
    context_policy=ContextPolicy(
        progressive_tool_disclosure=True,
        tool_top_k=4,
    ),
    retrieval_policy=RETRIEVAL_POLICY,
    tool_result_policy=TOOL_RESULT_POLICY,
    skill_retrieval=True,
    skill_retrieval_config=SKILL_RETRIEVAL_CONFIG,
    learning_control_plane=False,
    automations_enabled=False,
)
assert knowledge.attach_to_agent(support_agent, KB_ID)
support_agent.save()


### Inspect Toolbox routing and the before/after effect

The output includes Nova's refusal before registration, the strict JSON schema, governance policy, and the router's selected capability.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `semantic_tool_router.preview` | `query` | live health | Shows schema selection without executing the tool. |
| `semantic_tool_router.preview` | `user_id` | Maya | Applies user-aware routing scope. |
| tool metadata | `input_schema` | strict object | Rejects undeclared arguments. |
| prompt audit | `tool names` | before registration | Proves the callable was absent in the baseline. |

In [28]:
router_preview = support_agent.semantic_tool_router.preview(
    "check live service health", user_id=USER_ID
)
print("Before trusted capability:", before_tool)
print("Stored strict schema:", tool_metadata["input_schema"])
print("Stored policy:", tool_metadata["tool_policy"])
print("Progressive router preview:", router_preview)
toolbox_effect = {
    "no_callable_before": "service_health" not in before_tool_schemas,
    "schema_persisted": tool_metadata["name"] == "service_health",
    "router_selects_capability": "service_health" in router_preview,
}
print("Programmatic effect:", toolbox_effect)
assert all(toolbox_effect.values())
assert "service_health" in router_preview


Before trusted capability: Live health cannot be checked from the tools available in this session: no `service_health` tool is available.

Evidence boundary: supplied memory/context does not include current telemetry for `checkout-api`, and I won’t infer live state from stored memory.
Stored strict schema: {'type': 'object', 'properties': {'service': {'type': 'string', 'description': 'Parameter service'}}, 'additionalProperties': False, 'required': ['service']}
Stored policy: {'deterministic': True, 'side_effects': False, 'requires_approval': False, 'approval_reason': None, 'domains': ['health'], 'aliases': [], 'deprecated_arguments': {}}
Progressive router preview: ['service_health']
Programmatic effect: {'no_callable_before': True, 'schema_persisted': True, 'router_selects_capability': True}


## 8 · `TOOL_LOG`: full audit evidence without prompt bloat

The tool returns a deliberately long diagnostic list. This experiment first
raises the inline threshold for two calls, producing clean one-step trajectories
for a clean workflow comparison. It then restores the 320-character policy for a third
call: MemoRizz stores the complete result in `TOOL_LOG` and sends only a compact
digest/pointer through the model loop. The answer is based on live output, while
the full result remains recoverable by its logical `tool_log_id`.

This distinction is important: under the current policy, a small inline result
need not create a tool-log row. Offloading is both the storage trigger and the
prompt-size control demonstrated here. We inspect the actual model prompt to
prove that it received a pointer instead of the 24-row diagnostic payload.

Tool logs are operational memory: arguments, result, success/error, call ID,
timestamp, and exact scope. They are not a substitute for entity facts (stable
truth) or conversation history (what the user saw).

### Create repeatable inline live-health trajectories

A temporary high threshold keeps the full result inline so two differently phrased requests produce clean one-tool workflows.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `ToolResultPolicy` | `offload_above_chars` | `100000` | Temporarily prevents pointer creation. |
| `MemAgent.run` | `message` | use only `service_health` | Constrains the learned trajectory to one tool. |
| `MemAgent.run` | `thread_id` | two distinct threads | Creates independent workflow records. |

In [29]:
INLINE_TOOL_RESULT_POLICY = ToolResultPolicy(
    offload_above_chars=100_000,
    expansion_tool_names=TOOL_RESULT_POLICY.expansion_tool_names,
)
support_agent.tool_result_policy = INLINE_TOOL_RESULT_POLICY

first_live_answer = support_agent.run(
    "Use only service_health to check checkout-api live health. Summarize the result.",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="tool-run-one",
)
second_live_answer = support_agent.run(
    "Use only service_health to inspect checkout-api before escalation. Summarize the result.",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="tool-run-two",
)

print("Inline run 1:", first_live_answer)
print("Inline run 2:", second_live_answer)


Inline run 1: checkout-api live health (from `service_health` only):

- Status: degraded
- Deployment: blue-2026-08-25
- p95 latency: 480 ms
- Error rate: 2.2%
- Diagnostics: all 24 probes report elevated latency

Evidence boundary: I only used the live `service_health` result for this summary.
Inline run 2: checkout-api health check via `service_health`:

- Status: degraded
- Deployment: `blue-2026-08-25`
- p95 latency: 480 ms
- Error rate: 2.2%
- Diagnostics: repeated elevated latency across probes `00–23`

Escalation context from supplied memory: tier-1 service, owner Maya Chen, channel `#checkout-incidents`.


### Restore offloading and run from the compact digest

The normal 320-character policy stores the full result and sends GPT a digest plus logical tool-log pointer.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| agent policy | `tool_result_policy` | normal policy | Re-enables prompt-size control. |
| `MemAgent.run` | `thread_id` | `tool-log-offload` | Makes the offloaded call easy to locate. |
| prompt slice | `start index` | before this run | Audits only the messages generated by this experiment. |

In [30]:
# Restore normal prompt compaction and capture exactly the prompts from this run.
support_agent.tool_result_policy = TOOL_RESULT_POLICY
offload_prompt_start = len(teaching_model.prompts)
offloaded_live_answer = support_agent.run(
    "Use service_health to check checkout-api. Answer from its compact digest; "
    "do not expand the full diagnostics.",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="tool-log-offload",
)
offload_prompts = teaching_model.prompts[offload_prompt_start:]
offload_prompt_text = "\n".join(item["text"] for item in offload_prompts)

print("Offloaded run:", offloaded_live_answer)


Offloaded run: checkout-api is **degraded**.

Evidence from compact `service_health` digest:
- Status: **degraded**
- p95 latency: **480 ms**
- Error rate: **2.2%**
- Deployment: **blue-2026-08-25**
- Diagnostics shown in digest: latency elevated across probes 00–07, with additional diagnostics truncated.

I did **not** expand full diagnostics, per your instruction.


### Recover and verify the complete tool result

The model sees a pointer; application code can retrieve the full 24-row diagnostic result by its logical ID.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `list_tool_logs` | `memory_id / user_id` | current scope | Lists only authorized operational records. |
| `list_tool_logs` | `limit` | `10` | Bounds the audit query. |
| `retrieve_tool_log` | `tool_log_id` | offloaded call ID | Returns the complete stored result. |
| prompt audit | `pointer / final probe` | present / absent | Proves compaction rather than merely assuming it. |

In [31]:
tool_logs = support_agent.memory_manager.list_tool_logs(
    MEMORY_ID,
    user_id=USER_ID,
    limit=10,
)
health_logs = [row for row in tool_logs if row.get("tool_name") == "service_health"]
offloaded_health_logs = [
    row for row in health_logs if row.get("thread_id") == "tool-log-offload"
]
assert offloaded_health_logs, "The long live-health result was not offloaded."
newest_log = offloaded_health_logs[0]
log_record_id = str(
    newest_log.get("tool_log_id") or newest_log.get("_id") or newest_log.get("id")
)
restored_log = support_agent.memory_manager.retrieve_tool_log(
    log_record_id, user_id=USER_ID
)
full_result = json.loads(restored_log["result"])

print(
    "Audit:",
    {
        "tool_name": restored_log["tool_name"],
        "success": restored_log["success"],
        "result_chars": len(restored_log["result"]),
        "diagnostic_rows": len(full_result["diagnostics"]),
        "thread_id": restored_log["thread_id"],
        "pointer_seen_by_model": '"offloaded": true' in offload_prompt_text.lower(),
        "full_diagnostics_seen_by_model": "probe-23: latency elevated" in offload_prompt_text,
    },
)
assert full_result["status"] == "degraded" and len(full_result["diagnostics"]) == 24
assert '"offloaded": true' in offload_prompt_text.lower()
assert "probe-23: latency elevated" not in offload_prompt_text


Audit: {'tool_name': 'service_health', 'success': True, 'result_chars': 823, 'diagnostic_rows': 24, 'thread_id': 'tool-log-offload', 'pointer_seen_by_model': True, 'full_diagnostics_seen_by_model': False}


## 9 · `WORKFLOW_MEMORY`: what procedure actually ran?

Because workflow memory is active, each tool-calling turn automatically writes
a trajectory. MemoRizz canonicalizes ordered tool names, argument-key shapes,
errors, and collapsed retries. Literal argument values and results do not define
identity, so the two inline runs should share one trajectory class despite
different wording. The offloaded run can form another class if GPT chooses an
expansion tool; that difference is preserved rather than mislabeled.

Workflow memory is descriptive audit evidence: it records what happened, its
outcome, and its tool shape. MemoRizz deliberately does **not** inject raw past
workflows as instructions. A reusable procedure needs separate review and is
represented by `SKILLBOX` in the next section.


### Collect and group service-health workflows

Canonicalization groups procedures by ordered tool/argument shape rather than
literal query wording or result text.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `provider.list_all` | memory type | `WORKFLOW_MEMORY` | Provides the complete administrative audit set. |
| workflow filter | `agent / user / memory` | exact run | Removes unrelated trajectories. |
| `aggregate_trajectory_stats` | `agent_id / user_id` | Nova / Maya | Computes executions, outcome rate, and query diversity. |
| `Counter.most_common` | count | `1` | Selects the dominant health-check shape for readable comparison. |


In [32]:
all_agent_workflows = [
    row
    for row in (provider.list_all(MemoryType.WORKFLOW_MEMORY) or [])
    if row.get("agent_id") == AGENT_ID
    and row.get("user_id") == USER_ID
    and row.get("memory_id") == MEMORY_ID
]
workflow_rows = [
    row
    for row in all_agent_workflows
    if any(
        step.get("tool") == "service_health"
        for step in (row.get("canonical_signature") or [])
        if isinstance(step, dict)
    )
]
trajectory_stats = aggregate_trajectory_stats(
    provider,
    agent_id=AGENT_ID,
    user_id=USER_ID,
)
trajectory_counts = Counter(
    row.get("canonical_hash") for row in workflow_rows
)
HEALTH_TRAJECTORY_HASH, health_execution_count = (
    trajectory_counts.most_common(1)[0]
)
health_workflow_rows = [
    row
    for row in workflow_rows
    if row.get("canonical_hash") == HEALTH_TRAJECTORY_HASH
]
relevant_stats = [
    stat
    for stat in trajectory_stats
    if stat.canonical_hash == HEALTH_TRAJECTORY_HASH
]


### Print and validate workflow evidence

The output exposes every service-health query, canonical hash, step count, and
aggregate statistics. The assertions verify that differently worded requests
produced a repeatable procedure shape; they do not grant it instruction authority.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| workflow display | `canonical_hash` | first 12 characters | Makes grouping visible without overwhelming output. |
| trajectory stats | executions / success / diversity | runtime values | Distinguishes repetition, outcome, and wording diversity. |
| assertions | minimum evidence | 3 rows, repeated class, 2 queries | Protects the canonicalization demonstration from model variability. |


In [33]:
print(
    [
        {
            "query": row.get("user_query"),
            "outcome": row.get("outcome"),
            "step_count": row.get("step_count"),
            "canonical_hash": str(row.get("canonical_hash"))[:12],
            "steps": list((row.get("steps") or {}).keys()),
        }
        for row in workflow_rows
    ]
)
print(
    "Dominant trajectory statistics:",
    [
        {
            "executions": stat.executions,
            "success_rate": stat.success_rate,
            "distinct_queries": stat.distinct_query_count,
        }
        for stat in relevant_stats
    ],
)
print(
    "Canonical workflow effect:",
    {
        "canonical_hash": str(HEALTH_TRAJECTORY_HASH)[:12],
        "executions": health_execution_count,
        "queries": [row.get("user_query") for row in health_workflow_rows],
        "other_service_health_classes": len(trajectory_counts) - 1,
        "other_tool_bearing_workflows": (
            len(all_agent_workflows) - len(workflow_rows)
        ),
    },
)
assert len(workflow_rows) >= 3
assert health_execution_count >= 2
assert relevant_stats and relevant_stats[0].distinct_query_count >= 2
assert relevant_stats[0].success_rate == 1.0


[{'query': 'Use only service_health to inspect checkout-api before escalation. Summarize the result.', 'outcome': 'success', 'step_count': 1, 'canonical_hash': 'b0cb8d53d78b', 'steps': ['Step 1: service_health']}, {'query': 'Use only service_health to check checkout-api live health. Summarize the result.', 'outcome': 'success', 'step_count': 1, 'canonical_hash': 'b0cb8d53d78b', 'steps': ['Step 1: service_health']}, {'query': 'Use service_health to check checkout-api. Answer from its compact digest; do not expand the full diagnostics.', 'outcome': 'success', 'step_count': 1, 'canonical_hash': 'b0cb8d53d78b', 'steps': ['Step 1: service_health']}]
Dominant trajectory statistics: [{'executions': 3, 'success_rate': 1.0, 'distinct_queries': 3}]
Canonical workflow effect: {'canonical_hash': 'b0cb8d53d78b', 'executions': 3, 'queries': ['Use only service_health to inspect checkout-api before escalation. Summarize the result.', 'Use only service_health to check checkout-api live health. Summariz

## 10 · `SKILLBOX`: retrieve a reviewed reusable procedure

`WORKFLOW_MEMORY` showed what Nova actually did. `SKILLBOX` answers a different
question: which reviewed procedure is applicable to this request? To keep the
lesson focused, host code authors one procedure and marks it `ACTIVE` after
review. MemoRizz embeds its applicability fields—name, description,
preconditions, and example queries—with the configured OpenAI embedding model.

Only active, sufficiently similar skills are retrieved. The skill enters the
turn as user-authority context by default; it does not make its named tool
callable and it does not replace live telemetry. Automated candidate creation,
shadow testing, promotion, monitoring, and demotion belong in a dedicated
continual-learning lesson and are intentionally not run here.


### Establish the empty-Skillbox baseline

The same procedural question is asked before any reviewed skill exists. Direct
retrieval and the tracked prompt must both show that no Skillbox instruction was
available.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `retrieve_skills_by_query` | `query` | exact procedure question | Uses the real semantic retrieval path. |
| `retrieve_skills_by_query` | `user_id` | Maya | Enforces the skill's user boundary. |
| `MemAgent.run` | `thread_id` | `skill-before` | Isolates the baseline answer. |
| prompt audit | reviewed skill heading | absent | Proves no skill instructions reached GPT. |


In [34]:
skill_question = (
    "checkout-api is breaching latency. Do I have a reviewed reusable "
    "procedure for inspecting live health before escalation? Summarize it "
    "without executing any tools."
)
before_skill_hits = support_agent.skillbox.retrieve_skills_by_query(
    skill_question,
    limit=SKILL_RETRIEVAL_CONFIG["top_k"],
    min_similarity=SKILL_RETRIEVAL_CONFIG["min_similarity"],
    user_id=USER_ID,
)
before_skill_answer = support_agent.run(
    skill_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="skill-before",
)
prompt_before_skill = teaching_model.prompts[-1]["text"].lower()

display(Markdown(f"#### Before SKILLBOX memory\n\n{before_skill_answer}"))
skill_baseline_effect = {
    "retrieval_empty": not before_skill_hits,
    "reviewed_skill_absent_from_prompt": (
        "### reviewed live-health check" not in prompt_before_skill
    ),
}
print("Programmatic baseline:", skill_baseline_effect)
assert all(skill_baseline_effect.values())


#### Before SKILLBOX memory

I don’t have a reviewed reusable procedure in the supplied memory for inspecting `checkout-api` live health before escalation.

Evidence boundary:
- Supplied entity memory only lists `checkout-api` ownership/tier/region/escalation channel.
- No Skill Memory procedure was supplied in this turn.
- You explicitly asked not to execute tools, so I did not call `service_health` or any other tool.

Programmatic baseline: {'retrieval_empty': True, 'reviewed_skill_absent_from_prompt': True}


### Add one host-reviewed active skill

The procedure is deliberately authored by trusted host code rather than
generated from the preceding workflows. Workflow IDs and the canonical hash
are attached only as audit provenance. Creating the `Skill` also creates a real
hosted embedding through the global `EmbeddingManager`.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `Skill` | `name / description / content` | reviewed health check | Separates applicability text from procedure steps. |
| `Skill` | `queries / preconditions` | latency and escalation examples | Supplies the **when to apply** embedding signal. |
| `Skill` | `status` | `ACTIVE` | Makes the reviewed procedure eligible for retrieval. |
| `Skill` | `agent_id / user_id` | Nova / Maya | Prevents cross-agent or cross-user instruction leakage. |
| `Skillbox.add_skill` | skill | reviewed object | Persists the `SKILLBOX` representation. |


In [35]:
health_workflow_ids = [
    str(row.get("workflow_id") or row.get("_id") or row.get("id"))
    for row in health_workflow_rows
]
reviewed_skill = Skill(
    name="Reviewed live-health check",
    description=(
        "Use when a production service is slow or breaching latency and an "
        "operator needs fresh health evidence before escalation."
    ),
    content=(
        "1. Call `service_health` once for the named service.\n"
        "2. Report status, p95 latency, error rate, and deployment from that result.\n"
        "3. Compare telemetry with the attached approved runbook when available.\n"
        "4. State uncertainty explicitly and escalate when supplied evidence requires it."
    ),
    preconditions=[
        "A named production service is slow or breaching latency.",
        "Fresh telemetry is required before recommending escalation.",
    ],
    tools_used=["service_health"],
    queries=[
        skill_question,
        "Inspect live service health before escalation.",
        "Checkout-api is slow; check health before recommending next steps.",
    ],
    agent_id=AGENT_ID,
    user_id=USER_ID,
    source_canonical_hash=HEALTH_TRAJECTORY_HASH,
    source_workflow_ids=health_workflow_ids,
    exemplar_workflow_id=health_workflow_ids[0],
    status=SkillStatus.ACTIVE,
)
existing_skill = support_agent.skillbox.get_skill_by_name(reviewed_skill.name)
if existing_skill is None:
    skillbox_record_id = support_agent.skillbox.add_skill(reviewed_skill)
    active_skill = support_agent.skillbox.get_skill_by_id(reviewed_skill.skill_id)
else:
    skillbox_record_id = "already-present"
    active_skill = existing_skill

assert active_skill is not None
print(
    {
        "skillbox_record_id": str(skillbox_record_id),
        "skill_id": active_skill.skill_id,
        "status": active_skill.status.value,
        "embedding_dimensions": len(reviewed_skill.embedding),
        "source_workflows": len(active_skill.source_workflow_ids),
    }
)


{'skillbox_record_id': 'a06bb074-f8e3-4115-b817-eff461e3d376', 'skill_id': 'a06bb074-f8e3-4115-b817-eff461e3d376', 'status': 'active', 'embedding_dimensions': 256, 'source_workflows': 3}


### Retrieve the skill and show its effect on Nova

The exact question is repeated. The output prints both answers separately so
their bullet/list formatting remains valid, then audits semantic retrieval,
prompt injection, and the retained workflow provenance.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `retrieve_skills_by_query` | `min_similarity` | `0.70` | Rejects weak applicability matches. |
| `MemAgent.run` | same question | after skill write | Changes only the Skillbox state. |
| prompt audit | skill name | present | Proves the reviewed procedure reached GPT. |
| `retrieve_by_id` | source workflow IDs | exact lookup | Proves adding a skill did not erase audit evidence. |


In [36]:
active_hits = support_agent.skillbox.retrieve_skills_by_query(
    skill_question,
    limit=SKILL_RETRIEVAL_CONFIG["top_k"],
    min_similarity=SKILL_RETRIEVAL_CONFIG["min_similarity"],
    user_id=USER_ID,
)
after_skill_answer = support_agent.run(
    skill_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id="skill-after",
)
prompt_after_skill = teaching_model.prompts[-1]["text"]
source_workflow_rows = {
    workflow_id: provider.retrieve_by_id(
        workflow_id,
        MemoryType.WORKFLOW_MEMORY,
    )
    for workflow_id in active_skill.source_workflow_ids
}

display(Markdown(f"#### After reviewed SKILLBOX memory\n\n{after_skill_answer}"))
skill_effect = {
    "active": active_skill.status == SkillStatus.ACTIVE,
    "semantic_match": bool(active_hits),
    "injected_into_prompt": (
        active_skill.name.lower() in prompt_after_skill.lower()
    ),
    "procedure_visible_to_user": (
        "service_health" in after_skill_answer.lower()
        or "live health" in after_skill_answer.lower()
    ),
    "workflow_provenance_retained": all(source_workflow_rows.values()),
}
print(
    "Retrieved skill:",
    {
        "name": active_skill.name,
        "similarity": round(active_hits[0].similarity, 3) if active_hits else None,
        "injection_role": active_skill.injection_role.value,
    },
)
print("Programmatic before/after effect:", skill_effect)
assert active_hits[0].skill.skill_id == active_skill.skill_id
assert all(skill_effect.values())


#### After reviewed SKILLBOX memory

Yes — supplied memory includes a **reviewed reusable procedure** for inspecting live health before escalation.

**Reviewed procedure: live-health check**
Applies when a production service is slow or breaching latency and an operator needs fresh health evidence before escalation.

1. Call `service_health` once for the named service.
2. Report:
   - status
   - p95 latency
   - error rate
   - deployment
3. Compare the telemetry with the approved runbook, if one is attached.
4. State uncertainty explicitly.
5. Escalate when the supplied evidence requires it.

**Evidence boundary:** I did not execute any tools, per your request. This summary is based only on the reviewed procedure supplied in this turn’s memory/context.

Retrieved skill: {'name': 'Reviewed live-health check', 'similarity': 0.777, 'injection_role': 'user'}
Programmatic before/after effect: {'active': True, 'semantic_match': True, 'injected_into_prompt': True, 'procedure_visible_to_user': True, 'workflow_provenance_retained': True}


# Part IV · Working memory and reuse

## 11 · `SHORT_TERM_MEMORY`: request context that should not become history

The current deployment selected in a UI is useful for one turn but dangerous as
a durable fact. We create a small agent whose only active type is short-term
memory, pass a `context` dictionary, then repeat the question without it in a
fresh thread.

The first answer sees the page context; the fresh thread does not. There are no
application-authored short-term rows. `MemAgent.run()` still writes its normal
user/assistant audit conversation, even when conversation retrieval is not an
active memory type. That distinction matters: if the assistant repeats a page
value, the value can become conversation history in the same thread. We use a
new thread to prove that the original request context itself did not persist.

This is the intended short-term contract: MemoRizz assembles bounded working
context from the request, retrieval, tools, and runtime state. Use
`ContextPolicy`, thread boundaries, and context-window stats rather than treating
short-term memory as an unbounded public scratchpad.

### Run a request-context-only agent in two threads

The first request receives UI page context; the second fresh thread receives none. Both answers are printed.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent` | `memory_types` | SHORT_TERM only | Isolates the working-context behavior. |
| `MemAgent` | `retrieval_policy` | `False` | Disables durable semantic retrieval. |
| `MemAgent.run` | `context` | deployment page dictionary | Supplies runtime-only request state. |
| `MemAgent.run` | `thread_id` | fresh second thread | Prevents the first answer from becoming same-thread recall. |

In [37]:
working_model = make_model()
working_agent = MemAgent(
    model=working_model,
    name=f"Nova Working Context {RUN_ID}",
    application_id=f"memory-types-working-{RUN_ID}",
    instruction="Use current request context only when it is supplied.",
    memory_provider=provider,
    memory_ids=[WORKING_MEMORY_ID],
    memory_types=[MemoryType.SHORT_TERM_MEMORY],
    retrieval_policy=False,
    context_policy=ContextPolicy(progressive_tool_disclosure=False),
    learning_control_plane=False,
    automations_enabled=False,
)
working_scope = {
    "memory_id": WORKING_MEMORY_ID,
    "user_id": USER_ID,
    "thread_id": "deployment-page",
}
with_request_context = working_agent.run(
    "Which deployment is currently selected?",
    **working_scope,
    context={
        "current_page": {"type": "deployment", "id": "blue-2026-08-25"},
        "current_deployment": "blue-2026-08-25",
        "task_note": "compare this deployment with live telemetry",
    },
)
prompt_with_request_context = working_model.prompts[-1]["text"].lower()
without_context_scope = {
    **working_scope,
    "thread_id": "deployment-page-next",
}
without_request_context = working_agent.run(
    "Which deployment is currently selected?",
    **without_context_scope,
)
prompt_without_request_context = working_model.prompts[-1]["text"].lower()

print("With request context:", with_request_context)
print("Without request context:", without_request_context)


With request context: The currently selected deployment is `blue-2026-08-25`.
Without request context: I don’t have any deployment selection information in the current context, so I can’t tell which deployment is currently selected.


### Inspect short-term persistence and context statistics

Zero short-term rows is the intended result; conversation rows are a separate audit trail created by `run()`.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `list_all` | `memory type` | SHORT_TERM_MEMORY | Checks whether request context became a durable row. |
| conversation history | `thread_id` | each request thread | Separates runtime context from dialogue audit. |
| `get_context_window_stats` | `—` | latest turn | Shows prompt/completion token use. |

In [38]:
short_term_rows = [
    row
    for row in (provider.list_all(MemoryType.SHORT_TERM_MEMORY) or [])
    if row.get("memory_id") == WORKING_MEMORY_ID and row.get("user_id") == USER_ID
]
working_conversation = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=WORKING_MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id="deployment-page",
)
fresh_thread_conversation = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=WORKING_MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id="deployment-page-next",
)

print(
    {
        "short_term_rows": len(short_term_rows),
        "first_thread_conversation_rows": len(working_conversation),
        "fresh_thread_conversation_rows": len(fresh_thread_conversation),
        "context_window": working_agent.get_context_window_stats(),
    }
)
short_term_effect = {
    "request_context_injected": "blue-2026-08-25" in prompt_with_request_context,
    "request_context_expired": "blue-2026-08-25" not in prompt_without_request_context,
    "no_short_term_rows_persisted": not short_term_rows,
    "conversation_audit_is_separate": len(working_conversation) == 2
    and len(fresh_thread_conversation) == 2,
}
print("Programmatic effect:", short_term_effect)
assert all(short_term_effect.values())


{'short_term_rows': 0, 'first_thread_conversation_rows': 2, 'fresh_thread_conversation_rows': 2, 'context_window': {'timestamp': '2026-08-25T14:56:12.078987', 'prompt_tokens': 3192, 'completion_tokens': 60, 'total_tokens': 3252, 'context_window_tokens': 1050000, 'percentage_used': 0.3097142857142857, 'stage': 'iteration_1'}}
Programmatic effect: {'request_context_injected': True, 'request_context_expired': True, 'no_short_term_rows_persisted': True, 'conversation_audit_is_separate': True}


## 12 · `SEMANTIC_CACHE`: reuse only when scope and freshness agree

Semantic cache is an optimization, not durable knowledge. A hit requires a
similar query **and** matching agent/memory/user/session scope plus model,
prompt, tool-schema, completion-policy, data-version, and request-context
fingerprints. TTL and invalidation domains provide additional freshness.

The first runbook question is a miss and model call; the exact repeat is a hit
and makes no model call. Changing `data_version` forces another model call even
though the words are identical. We then invalidate version 2 and repopulate it
in a fresh session so the current version remains available.

We first retrieve the relevant runbook excerpt and pass it as versioned request
context, while removing the model-callable KB tool from this cache-only agent.
That keeps the cached operation a deterministic, side-effect-free reasoning
turn; a live or mutating tool loop would not be a safe admission candidate.

Side-effecting or non-deterministic operations should bypass admission. Never
use vector similarity alone as proof that an operational answer is still true.
The cell also retains physical row IDs before invalidation and applies an exact
`delete_by_id` compatibility cleanup when a backend's bulk invalidation removes
the hot entry but cannot enumerate its persistent semantic-cache row.

### Create a scoped semantic-cache agent

This agent answers a deterministic, read-only policy question. The KB tool is removed so cache behavior is not obscured by tool loops.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `MemAgent` | `memory_types` | KB and semantic cache | Limits the experiment to retrieval plus reuse. |
| semantic cache config | `similarity_threshold` | `0.99` | Requires an almost exact semantic match. |
| semantic cache config | `scope` | `session` | Prevents reuse across threads/sessions. |
| semantic cache config | `ttl_hours` | `1` | Adds a freshness deadline. |
| `RetrievalPolicy` | `conversation_scope` | `disabled` | Excludes dialogue history from the cache experiment. |

In [39]:
cache_model = make_model()
cache_agent = MemAgent(
    model=cache_model,
    name=f"Nova Policy Cache {RUN_ID}",
    application_id=f"memory-types-cache-{RUN_ID}",
    instruction="Answer deterministic read-only incident-policy questions from retrieved knowledge.",
    memory_provider=provider,
    memory_ids=[MEMORY_ID],
    memory_types=[MemoryType.KNOWLEDGE_BASE, MemoryType.SEMANTIC_CACHE],
    semantic_cache=True,
    semantic_cache_config={
        "similarity_threshold": 0.99,
        "scope": "session",
        "ttl_hours": 1,
    },
    retrieval_policy=RetrievalPolicy(
        conversation_scope="disabled",
        knowledge_base_scope="namespace",
        knowledge_base_namespaces=(KB_NAMESPACE,),
    ),
    context_policy=ContextPolicy(progressive_tool_disclosure=False),
    learning_control_plane=False,
    automations_enabled=False,
)
assert knowledge.attach_to_agent(cache_agent, KB_ID)
cache_agent.tool_manager.remove_tool("knowledge_base_lookup")


True

### Populate the cache, then repeat the exact request

`data_version` and invalidation domains become part of the cache safety context. Call-count deltas prove the second answer bypasses GPT.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| request context | `cache_domains` | `incident-policy` | Allows domain invalidation. |
| request context | `data_version` | `runbook-v1` | Fingerprints source freshness. |
| `MemAgent.run` | `same query and scope` | twice | Creates one miss/write followed by one hit. |
| model counter | `calls` | before/after | Measures whether GPT was invoked. |

In [40]:
cache_query = "What does a SEV-1 response team require?"
policy_excerpt = next(
    chunk["content"] for chunk in chunks if "incident commander" in chunk["content"]
)
context_v1 = {
    "cache_domains": ["incident-policy"],
    "data_version": "runbook-v1",
    "policy_excerpt": policy_excerpt,
}
cache_scope_v1 = {
    "memory_id": MEMORY_ID,
    "user_id": USER_ID,
    "thread_id": CACHE_THREAD_ID,
    "context": context_v1,
}
calls_before_cache = cache_model.calls
first_cache_answer = cache_agent.run(cache_query, **cache_scope_v1)
calls_after_first = cache_model.calls
second_cache_answer = cache_agent.run(cache_query, **cache_scope_v1)
calls_after_repeat = cache_model.calls


### Change the data version and inspect the new entry

The same text with a new source version must miss the v1 entry. The provider query captures the physical Oracle row needed for a documented compatibility cleanup.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| request context | `data_version` | `runbook-v2` | Forces a freshness-aware miss. |
| `inspect_semantic_cache` | `query / user / thread / context` | exact cache scope | Returns hit metadata without serving an answer. |
| `retrieve_by_query` | `memory type` | SEMANTIC_CACHE | Locates the persistent Oracle row. |
| `retrieve_by_query` | `agent/memory/session/user` | exact scope | Avoids an administrative cross-scope cache read. |

In [41]:
context_v2 = {
    "cache_domains": ["incident-policy"],
    "data_version": "runbook-v2",
    "policy_excerpt": policy_excerpt,
}
third_cache_answer = cache_agent.run(
    cache_query,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=CACHE_THREAD_ID,
    context=context_v2,
)
inspection_v2 = cache_agent.inspect_semantic_cache(
    cache_query,
    user_id=USER_ID,
    thread_id=CACHE_THREAD_ID,
    context=context_v2,
)
v2_candidates = provider.retrieve_by_query(
    query=cache_query,
    memory_store_type=MemoryType.SEMANTIC_CACHE,
    agent_id=cache_agent.agent_id,
    memory_id=MEMORY_ID,
    session_id=CACHE_THREAD_ID,
    user_id=USER_ID,
    limit=10,
)
v2_physical_ids = [
    str(row.get("_id"))
    for row in v2_candidates
    if row.get("cache_key") == inspection_v2.cache_key and row.get("_id")
]
calls_after_version = cache_model.calls
removed_v2 = cache_agent.invalidate_semantic_cache(data_version="runbook-v2")
provider_compat_cleanup = sum(
    bool(provider.delete_by_id(row_id, MemoryType.SEMANTIC_CACHE))
    for row_id in v2_physical_ids
)


### Invalidate and repopulate safely

High-level invalidation evicts the hot entry. The exact-ID deletion handles the current Oracle persistent-row compatibility edge, then a fresh session proves a new model call and write.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `invalidate_semantic_cache` | `data_version` | `runbook-v2` | Targets stale content by version. |
| `delete_by_id` | `physical cache row ID` | captured candidates | Removes the backend row when bulk enumeration misses it. |
| `MemAgent.run` | `thread_id` | fresh current session | Repopulates instead of reusing a stale session entry. |
| `semantic_cache_stats` | `—` | runtime counters | Exposes hits, misses, writes, evictions, and size. |

In [42]:
current_cache_thread = f"{CACHE_THREAD_ID}-current"
repopulated_cache_answer = cache_agent.run(
    cache_query,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=current_cache_thread,
    context=context_v2,
)
repopulated_inspection = cache_agent.inspect_semantic_cache(
    cache_query,
    user_id=USER_ID,
    thread_id=current_cache_thread,
    context=context_v2,
)
cache_stats = cache_agent.semantic_cache_stats()


### Print and assert cache evidence

The output shows exact answer equality, GPT call deltas, version miss, invalidation count, compatibility cleanup, repopulation, and final cache statistics.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| call counters | `exact repeat` | unchanged | Proves the cache served the second answer. |
| call counters | `version / repopulation` | increase | Proves freshness changes bypass reuse. |
| cache inspection | `hit` | `True` | Verifies the stored entry is queryable. |
| assertions | `answer content` | required SEV roles | Ensures a cached answer still makes sense. |

In [43]:
print(
    {
        "exact_repeat_equal": first_cache_answer == second_cache_answer,
        "model_calls_before": calls_before_cache,
        "model_calls_after_first": calls_after_first,
        "model_calls_after_exact_repeat": calls_after_repeat,
        "model_calls_after_version_change": calls_after_version,
        "model_calls_after_repopulation": cache_model.calls,
        "stats": {key: cache_stats.get(key) for key in ("hits", "misses", "writes", "evictions", "size")},
        "v2_inspection": inspection_v2.to_dict(),
        "invalidated_v2": removed_v2,
        "provider_compat_cleanup": provider_compat_cleanup,
        "repopulated_hit": repopulated_inspection.hit,
    }
)
assert first_cache_answer == second_cache_answer
assert "incident commander" in first_cache_answer.lower()
assert "communications lead" in first_cache_answer.lower()
assert calls_after_first > calls_before_cache
assert calls_after_repeat == calls_after_first
assert calls_after_version > calls_after_repeat
assert inspection_v2.hit and v2_physical_ids and removed_v2 >= 1
assert cache_model.calls > calls_after_version
assert repopulated_inspection.hit and cache_stats["hits"] >= 1


{'exact_repeat_equal': True, 'model_calls_before': 0, 'model_calls_after_first': 1, 'model_calls_after_exact_repeat': 1, 'model_calls_after_version_change': 2, 'model_calls_after_repopulation': 3, 'stats': {'hits': 1, 'misses': 3, 'writes': 3, 'evictions': 1, 'size': 1}, 'v2_inspection': {'lookup_query': 'What does a SEV-1 response team require?', 'hit': True, 'matched_query': 'What does a SEV-1 response team require?', 'cache_key': 'fed59bac-f143-5735-8f45-f35bacadb026', 'similarity': 1.0, 'ttl_seconds': 3600.0, 'expires_in_seconds': 3599.870073080063, 'age_seconds': 0.1299269199371338, 'hit_count': 0, 'bypass_reason': None, 'invalidation_domains': ['incident-policy'], 'invalidation_tags': []}, 'invalidated_v2': 1, 'provider_compat_cleanup': 1, 'repopulated_hit': True}


# Part V · Coordination and durable identity

## 13 · `SHARED_MEMORY`: a workflow-scoped coordination blackboard

The support agent now has enough evidence to escalate. A separate incident
commander should not receive every one of Nova's private stores. Shared memory
creates a narrow session with declared participants, workflow/user/trace scope,
and typed messages:

- `COMMAND` delegates work and names its target;
- `STATUS` reports progress/blockers;
- `REPORT` returns findings, gaps, citations, or summary links.

This is coordination state, not global knowledge. Sessions should have a bounded
lifetime, participant authorization, and tenant ownership checks.

### Create a scoped shared session and post a command

Participants share only the blackboard session, not Nova's private conversation/entity/knowledge stores.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `create_shared_session` | `root_agent_id` | Nova | Declares the coordinator. |
| `create_shared_session` | `delegate_agent_ids` | incident commander | Declares authorized participants. |
| `create_shared_session` | `workflow_id / user_id / trace_id` | run-scoped IDs | Adds workflow, tenant, and trace boundaries. |
| `post_command` | `target_agent_id` | incident commander | Directs work to one participant. |

In [44]:
shared = SharedMemory(provider)
INCIDENT_COMMANDER_ID = f"incident-commander-{RUN_ID}"
SHARED_ID = shared.create_shared_session(
    root_agent_id=AGENT_ID,
    delegate_agent_ids=[INCIDENT_COMMANDER_ID],
    workflow_id=SHARED_WORKFLOW_ID,
    user_id=USER_ID,
    trace_id=f"trace-{RUN_ID}",
)
shared.post_command(
    SHARED_ID,
    agent_id=AGENT_ID,
    command_id="assess-checkout",
    target_agent_id=INCIDENT_COMMANDER_ID,
    instructions=(
        "Confirm SEV-1 classification using the checkout runbook and review "
        "the latest service_health tool-log evidence."
    ),
)


True

### Post status/report entries and read the blackboard

Typed entries make delegation, progress, evidence, and remaining gaps separately inspectable.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `post_status` | `status / progress` | in progress / 50 | Reports execution state without pretending completion. |
| `post_report` | `findings` | live latency finding | Returns the delegate's conclusion. |
| `post_report` | `citations / gaps` | tool-log ID / duration gap | Preserves evidence and uncertainty. |
| `update_session_status` | `status` | completed | Closes the coordination lifecycle. |

In [45]:
shared.post_status(
    SHARED_ID,
    agent_id=INCIDENT_COMMANDER_ID,
    command_id="assess-checkout",
    status="in_progress",
    progress=50,
)
shared.post_report(
    SHARED_ID,
    agent_id=INCIDENT_COMMANDER_ID,
    command_id="assess-checkout",
    findings="Live p95 is 480 ms; continue timing the breach and prepare rollback.",
    citations=[str(newest_log.get("tool_log_id") or log_record_id)],
    gaps=["The 10-minute SEV-1 duration gate is not yet confirmed."],
)
shared.update_session_status(SHARED_ID, "completed")
shared_entries = shared.get_blackboard_entries(SHARED_ID)

print(
    [
        {
            "type": entry.get("entry_type"),
            "agent_id": entry.get("agent_id"),
            "payload": (entry.get("content") or {}).get("payload"),
        }
        for entry in shared_entries
    ]
)
assert [entry["entry_type"] for entry in shared_entries] == ["COMMAND", "STATUS", "REPORT"]


[{'type': 'COMMAND', 'agent_id': 'a62f6cee-d175-5b38-b969-510df5ae1852', 'payload': {'command_id': 'assess-checkout', 'target_agent_id': 'incident-commander-30e75d0141', 'instructions': 'Confirm SEV-1 classification using the checkout runbook and review the latest service_health tool-log evidence.', 'priority': 3, 'dependencies': [], 'metadata': {}}}, {'type': 'STATUS', 'agent_id': 'incident-commander-30e75d0141', 'payload': {'command_id': 'assess-checkout', 'agent_id': 'incident-commander-30e75d0141', 'status': 'in_progress', 'progress': 50, 'blockers': None, 'summary_ids': []}}, {'type': 'REPORT', 'agent_id': 'incident-commander-30e75d0141', 'payload': {'command_id': 'assess-checkout', 'agent_id': 'incident-commander-30e75d0141', 'findings': 'Live p95 is 480 ms; continue timing the breach and prepare rollback.', 'citations': ['92414b11-ea01-4a11-826a-ca5d6eb3a489'], 'gaps': ['The 10-minute SEV-1 duration gate is not yet confirmed.'], 'summary_ids': []}}]


## 14 · `MEMAGENT`: persist and restore the whole configured agent

`MEMAGENT` stores the durable definition that connects these memories: agent
identity, instruction, active memory types/IDs, persona snapshot, attached
knowledge IDs, toolbox/skill configuration, cache and policy metadata, and
application mode. Secrets and arbitrary executable code are not serialized.

There is one provider-specific edge worth making visible. MemoRizz 0.6's
current Oracle relational `personas` table stores the stable base projection
(`persona_id`, name, role, background, traits, expertise), but not the rich
`goals`, `version`, or `evolution_history` fields. Therefore `MemAgent.load()`
can restore the base attachment but cannot reconstruct Persona v2 from this
schema alone. The cell measures that limitation and then has trusted host code
rebind the reviewer-approved v2 snapshot captured at the governance boundary.
Document-oriented providers can retain a richer document shape; always verify
the selected provider's round-trip contract. A future Oracle migration should
add these fields before relying on versioned Persona restart continuity.

On reload the host supplies a model and the persisted toolbox metadata, then
explicitly re-registers the trusted Python callable with `ToolManager.add_tool`.
That is a security boundary, not an inconvenience: a database row describing a
callable must not become executable authority by itself. The audit verifies both
schema presence and an actual callable binding.

The final query reuses the original incident thread and asks for a live check.
The prompt audit then verifies that the restored turn contained evidence from
conversation/summary, persona, entity memory, knowledge, an active skill, and a
tool-log pointer.

### Save and reload the durable agent configuration

The provider restores identity, memory IDs, policies, attachments, and tool metadata. The host supplies a live model and trusted toolbox process object.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `support_agent.save` | `—` | current configuration | Writes the MEMAGENT representation. |
| `retrieve_memagent` | `agent_id` | `AGENT_ID` | Exposes the stored definition for audit. |
| `MemAgent.load` | `memory_provider` | Oracle | Reads the saved agent and attachments. |
| `MemAgent.load` | `model / toolbox` | runtime objects | Rehydrates non-secret, non-executable dependencies explicitly. |
| `MemAgent.load` | `automations_enabled` | `False` | Prevents background side effects during the lesson. |

In [46]:
support_agent.save()
stored_agent = provider.retrieve_memagent(AGENT_ID)
support_agent.close(close_memory_provider=False)

restored_model = make_model()
restored_agent = MemAgent.load(
    AGENT_ID,
    memory_provider=provider,
    model=restored_model,
    toolbox=toolbox,  # persisted schema/metadata; callable is rebound below
    automations_enabled=False,
)


### Rebind executable authority and reviewed Persona v2

Persisted schema is not callable authority. The cell also measures Oracle's base Persona projection before trusted host code rebinds the reviewed rich snapshot.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `tool_manager.add_tool` | `callable` | `service_health` | Restores executable code from trusted host code. |
| `get_tool_callable` | `name` | `service_health` | Verifies the binding exists. |
| `Persona.from_dict` | `snapshot` | reviewed v2 | Rehydrates without merging role defaults. |
| `set_persona` | `save` | `False` | Binds reviewed configuration without creating another Oracle row. |

In [47]:
# Persisted schema is metadata, not authority. Re-register the callable from
# trusted host code and verify the binding before allowing a live action.
restored_agent.tool_manager.add_tool(service_health)
service_health_callable = restored_agent.tool_manager.get_tool_callable(
    "service_health"
)
oracle_persona_projection = restored_agent.persona_manager.current_persona
oracle_projection_effect = {
    "base_name_restored": oracle_persona_projection.name == "Nova",
    "base_role_restored": oracle_persona_projection.role == RoleType.CUSTOMER_SUPPORT.value,
    "rich_version_restored": oracle_persona_projection.version == 2,
    "evolution_history_restored": bool(oracle_persona_projection.evolution_history),
}

# The reviewed snapshot is application-authoritative configuration. Rehydrate
# without re-merging role defaults, then bind it without creating another row.
reviewed_persona = Persona.from_dict(reviewed_persona_snapshot)
assert restored_agent.set_persona(reviewed_persona, save=False)
restored_persona = restored_agent.persona_manager.current_persona
print(
    {
        "stored_agent_id": stored_agent.agent_id,
        "restored_name": restored_agent.name,
        "memory_ids": restored_agent.memory_ids,
        "knowledge_base_ids": restored_agent.knowledge_base_ids,
        "oracle_persona_projection": oracle_projection_effect,
        "reviewed_persona_version_after_rebind": restored_persona.version,
        "reviewed_persona_goals_after_rebind": restored_persona.goals,
        "service_health_schema_present": "service_health" in restored_agent.tool_manager.list_tools(),
        "service_health_callable_rebound": service_health_callable is not None,
    }
)
assert restored_agent.agent_id == AGENT_ID
assert KB_ID in restored_agent.knowledge_base_ids
assert "service_health" in restored_agent.tool_manager.list_tools()
assert service_health_callable is not None
assert oracle_projection_effect["base_name_restored"]
assert oracle_projection_effect["base_role_restored"]
assert not oracle_projection_effect["rich_version_restored"]
assert not oracle_projection_effect["evolution_history_restored"]
assert restored_persona.version == 2
assert "state uncertainty explicitly" in restored_persona.goals.lower()


{'stored_agent_id': 'a62f6cee-d175-5b38-b969-510df5ae1852', 'restored_name': 'Nova Support 30e75d0141', 'memory_ids': ['nova-support-30e75d0141'], 'knowledge_base_ids': ['b92479e1-c55c-477a-8c96-34c648be1404'], 'oracle_persona_projection': {'base_name_restored': True, 'base_role_restored': True, 'rich_version_restored': False, 'evolution_history_restored': False}, 'reviewed_persona_version_after_rebind': 2, 'reviewed_persona_goals_after_rebind': 'Stay calm, evidence-first, and concise during production incidents; state uncertainty explicitly and never fabricate live state.', 'service_health_schema_present': True, 'service_health_callable_rebound': True}


# Part VI · Integrated proof, coverage, and cleanup

The separate representations now work together. A short grounding turn first
retrieves the exact policy evidence; the next turn carries it forward through
conversation memory while adding Persona stance, entity ownership, a reviewed
skill, fresh telemetry, and Maya's formatting preference.


### Ground the final update with approved policy

After reload, Nova first asks the exact policy question that passed Section 6.
Knowledge can reach the model through automatic semantic selection or through
the progressively disclosed `knowledge_base_lookup` tool. Because tool results
enter a later model step, this audit captures **all** prompts created by the
turn; `last_retrieval_evidence()` is printed separately as the automatic-
candidate diagnostic and is not treated as a tool-call ledger.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| prompt start index | `len(restored_model.prompts)` | before the turn | Isolates every model step in this request. |
| `restored_agent.run` | message | exact SLO and SEV-1 question | Gives retrieval a focused target. |
| `restored_agent.run` | `thread_id` | original incident thread | Creates a grounded conversational bridge. |
| `knowledge_base_ids` | membership | `KB_ID` | Proves the approved runbook attachment survived reload. |
| prompt audit | both thresholds | required | Proves runbook evidence reached GPT regardless of retrieval path. |


In [48]:
grounding_question = policy_question
grounding_prompt_start = len(restored_model.prompts)
grounding_answer = restored_agent.run(
    grounding_question,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
grounding_prompts = restored_model.prompts[grounding_prompt_start:]
grounding_prompt = "\n".join(item["text"] for item in grounding_prompts).lower()
grounding_retrieval = restored_agent.last_retrieval_evidence()
grounding_sources = {
    item.get("source") for item in grounding_retrieval["items"]
}
grounding_effect = {
    "approved_kb_attachment_restored": KB_ID in restored_agent.knowledge_base_ids,
    "slo_evidence_reached_gpt": "below 300 milliseconds" in grounding_prompt,
    "severity_evidence_reached_gpt": "10 continuous minutes" in grounding_prompt,
    "answer_uses_both_thresholds": (
        "300" in grounding_answer and "450" in grounding_answer
    ),
}

print("Grounding turn:", grounding_answer)
print("Automatic retrieval sources:", sorted(grounding_sources))
print("Programmatic grounding effect:", grounding_effect)
assert all(grounding_effect.values())


Grounding turn: - **Latency objective:** checkout-api production **p95 latency must stay below 300 ms**.
- **Declare SEV-1 when:** p95 latency **exceeds 450 ms for 10 continuous minutes**, or errors **exceed 5%**.

Evidence boundary: from retrieved KB `checkout-runbook-30e75d0141`. This does **not** indicate current/live checkout-api health.
Automatic retrieval sources: ['episodic']
Programmatic grounding effect: {'approved_kb_attachment_restored': True, 'slo_evidence_reached_gpt': True, 'severity_evidence_reached_gpt': True, 'answer_uses_both_thresholds': True}


### Run the integrated incident request

The second turn explicitly names the reviewed procedure intent so it satisfies
the Skillbox's instruction-safety threshold, then asks for live state and a
single user-facing update.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `retrieve_skills_by_query` | final request | same text sent to Nova | Previews the strict semantic match after reload. |
| skill retrieval | `min_similarity` | `0.70` | Keeps the instruction-bearing retrieval policy unchanged. |
| `restored_agent.run` | `thread_id` | same incident thread | Carries the grounded policy answer forward. |
| `restored_agent.run` | message | integrated incident request | Adds Persona, entity facts, reviewed procedure, and live health. |


In [49]:
final_request = (
    "Inspect live service health before escalation because checkout-api is "
    "breaching latency. Use the reviewed reusable procedure, then give me an "
    "incident update in my preferred format with the owner, latency objective, "
    "live state, and recommended next steps."
)
final_skill_hits = restored_agent.skillbox.retrieve_skills_by_query(
    final_request,
    limit=SKILL_RETRIEVAL_CONFIG["top_k"],
    min_similarity=SKILL_RETRIEVAL_CONFIG["min_similarity"],
    user_id=USER_ID,
)
final_answer = restored_agent.run(
    final_request,
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
final_prompt = restored_model.prompts[-1]["text"].lower()
retrieval_evidence = restored_agent.last_retrieval_evidence()
bullet_lines = [
    line
    for line in final_answer.splitlines()
    if re.match(r"^\s*(?:[-*•]|\d+[.)])\s+", line)
]


### Audit the integrated result

The full answer is printed along with strict skill similarity, prompt markers,
retrieval sources, and preferred-format evidence.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| prompt evidence | six markers | conversation, Persona, entity, policy, skill, tool log | Proves the layers reached GPT. |
| skill match | similarity | runtime score | Shows why the reviewed procedure passed policy. |
| answer audit | bullet count | 1 through 5 | Checks Maya's user-visible format preference. |
| assertions | all evidence | required | Makes a plausible answer insufficient without memory causality. |


In [50]:
prompt_evidence = {
    "conversation_or_summary": "under five bullets" in final_prompt,
    "persona": "state uncertainty explicitly" in final_prompt,
    "entity": "maya chen" in final_prompt and "#checkout-incidents" in final_prompt,
    "grounded_policy_continuity": (
        ("300 ms" in final_prompt or "300 milliseconds" in final_prompt)
        and ("450 ms" in final_prompt or "450 milliseconds" in final_prompt)
    ),
    "skill": active_skill.name.lower() in final_prompt,
    "tool_log_pointer": "tool_log_id" in final_prompt or "offloaded" in final_prompt,
}

print("Final integrated answer:", final_answer)
print(
    "Final reviewed-skill match:",
    round(final_skill_hits[0].similarity, 3) if final_skill_hits else None,
)
print("Prompt evidence:", prompt_evidence)
print("Preferred-format bullet count:", len(bullet_lines))
print(
    "Final-turn retrieval sources:",
    sorted({item.get("source") for item in retrieval_evidence["items"]}),
)
assert final_skill_hits[0].skill.skill_id == active_skill.skill_id
assert all(prompt_evidence.values())
assert isinstance(final_answer, str) and final_answer.strip()
assert 1 <= len(bullet_lines) <= 5


Final integrated answer: - **Owner / service:** checkout-api is owned by **Maya Chen**; tier-1, eu-west, escalation channel **#checkout-incidents**.
- **Latency objective:** production p95 latency must stay **<300 ms**; live p95 is **480 ms**, so it is breaching. Live status: **degraded**, error rate **2.2%**, deployment **blue-2026-08-25**.
- **SEV-1 check:** p95 is above the **450 ms** SEV-1 threshold, but supplied live health does not prove **10 continuous minutes**; errors are below the **5%** SEV-1 threshold.
- **Recommended next steps:** page you now before any executive email, post/update **#checkout-incidents**, engage Maya Chen, and verify whether p95 has exceeded 450 ms for 10 continuous minutes; declare SEV-1 if confirmed.

Evidence boundary: live telemetry is from `service_health(checkout-api)` this turn; owner/escalation facts are from supplied entity memory; latency/SEV thresholds are from prior supplied runbook context in this conversation.
Final reviewed-skill match: 0.

## Coverage audit: every type demonstrated, no semantic shortcuts

The table below is generated from the runtime and provider evidence. A non-zero
count is expected for durable stores. Short-term memory intentionally reports
zero persisted rows because its demonstrated effect came from request context.

This is a coverage assertion, not a benchmark. Production evaluation still needs
retrieval recall/precision, grounded-answer accuracy, freshness incidents,
p50/p95 latency, token/cost budgets, cache safety, and workflow outcome quality.

### Collect evidence counts for all thirteen types

Counts come from the runtime or exact provider scopes. Short-term memory is the intentional zero because its effect is request-local.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| conversation history | `scope` | original incident | Counts episodic rows used for continuity. |
| `list_all` filters | `agent_id / user_id` | current run | Counts workflows and toolbox metadata without cross-run contamination. |
| `list_tool_logs` | `limit` | `100` | Counts offloaded operational evidence. |
| `semantic_cache_stats` | `size` | runtime cache | Counts the repopulated cache entry. |

In [51]:
scoped_conversation_count = len(
    provider.retrieve_conversation_history_ordered_by_timestamp(
        memory_id=MEMORY_ID,
        memory_type=MemoryType.CONVERSATION_MEMORY,
        user_id=USER_ID,
        thread_id=THREAD_ID,
    )
)
scoped_workflows = [
    row
    for row in (provider.list_all(MemoryType.WORKFLOW_MEMORY) or [])
    if row.get("agent_id") == AGENT_ID and row.get("user_id") == USER_ID
]
scoped_tool_logs = restored_agent.memory_manager.list_tool_logs(
    MEMORY_ID, user_id=USER_ID, limit=100
)
scoped_toolbox = [
    row
    for row in (provider.list_all(MemoryType.TOOLBOX) or [])
    if row.get("agent_id") == AGENT_ID and row.get("user_id") == USER_ID
]
semantic_cache_count = cache_agent.semantic_cache_stats()["size"]


### Render and assert the complete coverage table

The generated table pairs each enum/store value with a count and observable effect, then asserts exact enum coverage.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| coverage list | `MemoryType` | all 13 enum values | Prevents accidental omission of a library memory type. |
| Markdown table | `evidence count` | runtime value | Makes coverage human-readable. |
| assertions | `durable counts` | greater than zero | Requires real stored evidence. |
| assertion | `SHORT_TERM count` | zero | Requires the request-local contract demonstrated earlier. |

In [52]:
coverage = [
    (MemoryType.PERSONAS, 1 if Persona.retrieve_persona(PERSONA_STORAGE_ID, provider) else 0, "stance changed; Oracle base projection measured; reviewed v2 rebound"),
    (MemoryType.TOOLBOX, len(scoped_toolbox), "strict service_health schema + trusted binding"),
    (MemoryType.ENTITY_MEMORY, len(entities.list_entities(memory_id=MEMORY_ID, user_id=USER_ID)), "owner/channel became available"),
    (MemoryType.SHORT_TERM_MEMORY, len(short_term_rows), "0 rows by design; request context absent in a fresh thread"),
    (MemoryType.KNOWLEDGE_BASE, len(knowledge.retrieve_knowledge(KB_ID)), "SLO/SEV-1 passages retrieved"),
    (MemoryType.CONVERSATION_MEMORY, scoped_conversation_count, "preference recalled in exact thread"),
    (MemoryType.WORKFLOW_MEMORY, len(scoped_workflows), "tool trajectories canonicalized"),
    (MemoryType.SKILLBOX, len(restored_agent.skillbox.list_skills()), "reviewed authored skill retrieved and injected"),
    (MemoryType.MEMAGENT, 1 if provider.retrieve_memagent(AGENT_ID) else 0, "agent restored with attachments"),
    (MemoryType.SHARED_MEMORY, len(shared_entries), "command/status/report hand-off"),
    (MemoryType.SUMMARIES, len(summary_ids), "source-linked compaction"),
    (MemoryType.SEMANTIC_CACHE, semantic_cache_count, "hit, version miss, invalidation, repopulation"),
    (MemoryType.TOOL_LOG, len(scoped_tool_logs), "full live result recoverable by ID"),
]

table = ["| Enum | Store value | Evidence count | Observable effect |", "|---|---|---:|---|"]
table.extend(
    f"| `{memory_type.name}` | `{memory_type.value}` | {count} | {effect} |"
    for memory_type, count, effect in coverage
)
display(Markdown("\n".join(table)))

covered = {memory_type for memory_type, _, _ in coverage}
assert covered == set(MemoryType)
assert all(count > 0 for memory_type, count, _ in coverage if memory_type != MemoryType.SHORT_TERM_MEMORY)
assert next(count for memory_type, count, _ in coverage if memory_type == MemoryType.SHORT_TERM_MEMORY) == 0


| Enum | Store value | Evidence count | Observable effect |
|---|---|---:|---|
| `PERSONAS` | `personas` | 1 | stance changed; Oracle base projection measured; reviewed v2 rebound |
| `TOOLBOX` | `toolbox` | 1 | strict service_health schema + trusted binding |
| `ENTITY_MEMORY` | `entity_memory` | 1 | owner/channel became available |
| `SHORT_TERM_MEMORY` | `short_term_memory` | 0 | 0 rows by design; request context absent in a fresh thread |
| `KNOWLEDGE_BASE` | `knowledge_base` | 4 | SLO/SEV-1 passages retrieved |
| `CONVERSATION_MEMORY` | `conversation_memory` | 13 | preference recalled in exact thread |
| `WORKFLOW_MEMORY` | `workflow_memory` | 9 | tool trajectories canonicalized |
| `SKILLBOX` | `skillbox` | 1 | reviewed authored skill retrieved and injected |
| `MEMAGENT` | `agents` | 1 | agent restored with attachments |
| `SHARED_MEMORY` | `shared_memory` | 3 | command/status/report hand-off |
| `SUMMARIES` | `summaries` | 1 | source-linked compaction |
| `SEMANTIC_CACHE` | `semantic_cache` | 1 | hit, version miss, invalidation, repopulation |
| `TOOL_LOG` | `tool_log` | 2 | full live result recoverable by ID |

## Complete operation and lifecycle matrix

| Type | Normal writer | Normal read/use | Update / lifecycle | Common failure to avoid |
|---|---|---|---|---|
| `CONVERSATION_MEMORY` | `agent.run()` | exact scoped history + bounded semantic recall | summary marker, retention deletion | using an unscoped transcript read in a user request |
| `SUMMARIES` | `generate_summaries()` / compaction policy | compact registry, `fetch_context_summary`, expansion by source IDs | regenerate or delete under retention policy | treating compression as source deletion |
| `PERSONAS` | host or reviewed persona tool | stable system-prompt identity | versioned `Persona.update()` with trigger | storing customer facts in the agent's identity |
| `ENTITY_MEMORY` | `upsert_entity`, `record_attribute`, governed tools | exact ID/name or scoped semantic profile | source-aware attribute correction/removal | persisting volatile telemetry as durable truth |
| `KNOWLEDGE_BASE` | text/file/directory ingestion | namespace/tenant-scoped chunk retrieval | re-ingest/version source; detach separately from delete | attaching a KB and forgetting retrieval scope |
| `TOOLBOX` | trusted host registration | progressive schema discovery + in-process callable | deprecate aliases/arguments; rebind after restart | treating persisted schema as execution authority |
| `WORKFLOW_MEMORY` | automatic tool loop | audit aggregation, not automatic prompt recall | outcome annotation, canonical grouping, retention | interpreting repetition as reviewed instructions |
| `SKILLBOX` | trusted host/reviewer | active-skill applicability retrieval | revise/version/deprecate under review | assuming stored workflow history is already a reusable skill |
| `TOOL_LOG` | tool loop | digest in context; full result by scoped ID | retention/offload policy | copying huge results into every later prompt |
| `SHORT_TERM_MEMORY` | runtime context assembler | current request/context stats | expires with turn/task budget | exposing an unbounded mutable scratchpad API |
| `SEMANTIC_CACHE` | cache admission after safe response | scoped fresh lookup/inspection | TTL, domain/tag/version invalidation, eviction | confusing semantic similarity with freshness |
| `SHARED_MEMORY` | orchestrator and authorized participants | workflow-scoped blackboard | session status/retention | one global multi-tenant coordination bucket |
| `MEMAGENT` | `save`, auto-registration, builder `build_and_save` | `retrieve_memagent`, `MemAgent.load` | save refreshed config; delete agent scope | serializing secrets or trusting arbitrary code from storage |

## Key takeaways from the connected incident

1. **Start with the question a representation answers.** “What did Maya say?”
   is episodic; “who owns the service?” is entity memory; “what does policy
   say?” is knowledge; “what is happening now?” requires a live tool.
2. **Keep ownership explicit.** Host code owns scope, credentials, trusted
   callables, fact sources, review decisions, and lifecycle policy. The model
   does not grant its own output execution or instruction authority.
3. **Write rich evidence, retrieve bounded context.** Durable history, tool
   logs, and workflows can remain complete while prompts contain only selected,
   deduplicated, source-labelled evidence.
4. **A workflow and a skill are not the same thing.** Workflow memory describes
   what ran. Skillbox contains a reviewed procedure that may guide a future
   turn. This lesson authors that procedure directly and keeps automated
   continual learning outside its scope.
5. **Freshness needs a mechanism.** Entity provenance, document versions, live
   tools, and cache TTL/data versions solve different forms of staleness.
   Similarity alone solves none of them.
6. **Scope every boundary.** `memory_id`, `user_id`, `thread_id`, namespace,
   agent ownership, shared-session participants, and cache session are part of
   correctness and security—not optional metadata.
7. **Test round trips and user-visible effects.** A successful write is not
   enough. This notebook inspects provider rows, retrieval results, prompts,
   printed agent answers, reload behavior, and cleanup.


## Cleanup

Each run used unique identifiers. Unless `MEMORIZZ_TUTORIAL_KEEP_DATA=1`, the
final cell deletes knowledge chunks, the shared session, both user scopes,
cache/working scopes, persona, toolbox/skill/agent-owned records, and saved agent
definitions. Cleanup is explicit because content rows and agent-owned metadata
do not all share one physical key.

### Close agents, delete exact scopes, and verify cleanup

Cleanup is explicit because knowledge chunks, shared sessions, content scopes, Persona rows, and agent-owned configuration use different identifiers.

| Call or object | Parameter / argument | Value used here | Why it matters |
|---|---|---|---|
| `agent.close` | `close_memory_provider` | `False` | Closes runtimes while keeping the shared provider available for deletion. |
| `delete_knowledge` | `KB_ID` | this runbook | Deletes all four chunks. |
| `delete_scope` | `memory_id / user_id` | exact run scopes | Deletes content, shared, other-user, and working records. |
| `delete_scope` | `agent_ids` | three runtime agents | Deletes toolbox, skills, workflows, caches, and saved definitions. |
| `provider.close` | `—` | after verification | Releases Oracle resources only after assertions pass. |

In [53]:
agent_objects = [restored_agent, working_agent, cache_agent]
agent_ids = sorted({agent.agent_id for agent in agent_objects} | {AGENT_ID})
for agent in agent_objects:
    agent.close(close_memory_provider=False)

if KEEP_DATA:
    print(
        {
            "kept": True,
            "memory_id": MEMORY_ID,
            "working_memory_id": WORKING_MEMORY_ID,
            "user_id": USER_ID,
            "agent_ids": agent_ids,
            "knowledge_base_id": KB_ID,
            "shared_memory_id": SHARED_ID,
        }
    )
else:
    knowledge_deleted = knowledge.delete_knowledge(KB_ID)
    shared_cleanup = provider.delete_scope(memory_id=SHARED_ID)
    persona_deleted = Persona.delete_persona(PERSONA_STORAGE_ID, provider)
    content_cleanup = provider.delete_scope(memory_id=MEMORY_ID, user_id=USER_ID)
    other_user_cleanup = provider.delete_scope(memory_id=MEMORY_ID, user_id=OTHER_USER_ID)
    working_cleanup = provider.delete_scope(memory_id=WORKING_MEMORY_ID, user_id=USER_ID)
    agent_cleanup = provider.delete_scope(agent_ids=agent_ids)

    assert not knowledge.retrieve_knowledge(KB_ID)
    assert Persona.retrieve_persona(PERSONA_STORAGE_ID, provider) is None
    assert entities.list_entities(memory_id=MEMORY_ID, user_id=USER_ID) == []
    assert provider.retrieve_memagent(AGENT_ID) is None
    print(
        {
            "knowledge_deleted": knowledge_deleted,
            "persona_deleted": persona_deleted,
            "shared_deleted": shared_cleanup.get("total_deleted"),
            "content_deleted": content_cleanup.get("total_deleted"),
            "other_user_deleted": other_user_cleanup.get("total_deleted"),
            "working_deleted": working_cleanup.get("total_deleted"),
            "agent_owned_deleted": agent_cleanup.get("total_deleted"),
            "verified_remaining_entities": 0,
        }
    )

provider.close()
print("Notebook completed end to end; Oracle resources are closed.")


{'knowledge_deleted': True, 'persona_deleted': True, 'shared_deleted': 1, 'content_deleted': 61, 'other_user_deleted': 3, 'working_deleted': 5, 'agent_owned_deleted': 79, 'verified_remaining_entities': 0}


Notebook completed end to end; Oracle resources are closed.
